# AI ANALYST LAB

![](../_img/ai_analyst_lab.png)

### A Hands-on Course on AI for Data Analysts
## Session 04: Drivers analysis with correlation and regression

Feedback should be sent to [goran.milovanovic@datakolektiv.com](mailto:goran.milovanovic@datakolektiv.com).

This notebook accompanies the **AI Analyst LAB** course. Welcome to Session 04 — your first week building drivers-analysis muscle with correlation and regression on real wine-chemistry data.


***
### What we will do today

In the first three sessions we built the analyst's foundation: descriptive statistics and the sampling distribution of the mean (Session 01), conditional probability, expected value, and the bootstrap (Session 02), and the hypothesis-testing framework with chi-square and t-tests (Session 03). This week we cross from "describing a single number" into **describing relationships between two or more numbers** — the engine room of *drivers analysis*.

The business setting is a **mid-sized winery's quality team**. The head winemaker scores every batch on a 0–10 quality panel after bottling — but by then it is too late to change anything. The eleven measurements taken during fermentation are the only knobs she can actually turn. Her question is the most common analyst question in the world: **"Which of these inputs actually move the outcome, and how confident are we?"** We will answer it honestly, step by step, building the machinery from the ground up.

The sections, in order:

| Section | What happens |
|---|---|
| 4.1 | The business case — the winemaker's brief |
| 4.2 | Meet your Session 04 Tutor (Claude Project) |
| 4.3 | Setup — imports and loading the wine data |
| 4.4 | Terminology — *dependent / independent / criterion / predictor / outcome / feature / endogenous / exogenous* |
| 4.5 | First look — visual exploration in multi-panel plots |
| 4.6 | **Covariance** — measuring how two variables move together |
| 4.7 | **Correlation** — covariance, rescaled to $[-1, +1]$ |
| 4.8 | Correlation matrix and ranking the candidate drivers |
| 4.9 | **Simple linear regression** — the model |
| 4.10 | The residual scatterplot — the single most important picture in this notebook |
| 4.11 | **$R^2$** — what the line explains, what it does not |
| 4.12 | **Partial correlation** — built from residuals |
| 4.13 | **Part (semi-partial) correlation** — the *"all-else-equal"* primitive |
| 4.14 | **Multiple linear regression** — many drivers at once, with VIF and diagnostics |
| 4.15 | **Dummy coding** — adding a categorical predictor (red vs. white) |
| 4.16 | Our API calls — Anthropic tool use for the drivers memo |
| 4.17 | The drivers plan and stakeholder memo — fully worked |
| 4.18 | References — what to study to deepen this session |

A few rules for using this notebook, same as the previous three weeks:

- **Run the cells in order.** Each section builds on the one before.
- **Read the explanations, do not just run the cells.** The intuition is the point; the formulas are the receipts.
- **Every line of code has a comment above it** in beginner language.
- **Use your Session 04 Tutor** (the Claude Project at `_tutors/session04_tutor.xml`) when something feels confusing.
- **Compute first in Python, then use the model to interpret.** Same rule as Sessions 01–03 — the model never invents numbers.


***
## 4.1 The business case

You have just moved teams again. This week you are embedded with the **quality team at a mid-sized red-wine producer**. Every bottle that leaves the cellar has been scored on a 0–10 quality panel — three sensory experts taste, compare notes, and agree on a single integer score for the batch. That score is the team's headline KPI: it correlates with retail price, distributor demand, and the "drinking now / age it longer" call.

Your manager — the head winemaker — drops a USB stick on your desk on a Monday morning and says:

> *"We score every batch after bottling, but by then it is too late to change anything. During fermentation we already take eleven physicochemical measurements on every batch — pH, alcohol, residual sugar, the sulphates, all of it. I want to know which of those measurements actually **move** the panel score, and how confident we are. The picture I need by Friday: a ranked list of drivers, a fitted model that explains as much of the panel-score variance as it honestly can, and a one-page memo I can show the owner. Be straight with me — if a relationship is small or shaky, say so. I would rather hear 'we cannot tell' than 'we can' followed by a backslide three vintages from now."*

This is the canonical **drivers analysis** brief, and it is the backbone of many analyst roles. *"What drives revenue?"* *"What drives churn?"* *"What drives time-on-page?"* — every one of them lives in the same conceptual machinery we will build today.

By the end of this session you will have:

1. **Loaded** the UCI Wine Quality red-wine dataset and done the data-quality sanity-check discipline we built in Sessions 01–03.
2. **Mapped the terminology landscape** — *dependent / independent / criterion / predictor / outcome / feature / endogenous / exogenous* — so you can read papers, textbooks, and stakeholder emails from any tradition without confusion.
3. **Built covariance from scratch** as a measure of how two variables move together.
4. **Derived Pearson correlation** as standardised covariance — and seen, by direct computation, that $r = \mathrm{cov}(z_X, z_Y)$.
5. **Computed and interpreted a correlation matrix**, ranked the candidate drivers, and visualised the ranking.
6. **Fit a simple linear regression**, read every line of its summary table, drawn **the residual scatterplot** — the single most important figure in the notebook — and understood the **t-test on a regression coefficient** as *"how many standard errors does our observed slope sit away from the null value of zero?"*.
7. **Computed $R^2$ from scratch** as a variance-accounting decomposition, and seen that for one predictor $R^2 = r^2$ exactly.
8. **Built partial and part (semi-partial) correlations** from residuals — the conceptual primitive that makes a multiple regression's *"all else equal"* claim precise rather than mysterious.
9. **Fit a multiple regression**, read its summary, ranked its standardised coefficients, computed **VIF** to check for multicollinearity, and inspected diagnostic plots.
10. **Extended the multiple regression to include a categorical predictor** (red vs. white wines) via **dummy coding**, and interpreted the categorical coefficient as an *all-else-equal* difference against a chosen baseline.
11. **Generated a structured drivers plan via Anthropic tool use** (the §3.12 pattern, applied to this session's deliverable), and **drafted the stakeholder memo paragraph** from numbers Python computed.

Two threads weave through the whole session:

- **Honest interpretation of relationships** — correlation is association, never causation; a regression coefficient under multiple regression is an *"all else equal"* claim that needs care.
- **The same conceptual primitive — residuals — keeps coming back.** $R^2$, partial correlation, part correlation, multiple regression's *"all else equal"*, the dummy coefficient's vertical-gap interpretation, the diagnostic plots — all of them are built on what the model **fails** to predict. The residual is the unit of currency.

The artifacts in §4.17 bring both threads together. Let us get started.


***
## 4.2 Meet your Session 04 Tutor (Claude Project)

Before you continue, you should have set up your **Session 04 Tutor** — a Claude Project configured to teach you the correlation-and-regression ideas in this notebook in a gentle, beginner-friendly way.

If you have not done this yet, open **[`_tutors/TutorProjectCreation.md`](../_tutors/TutorProjectCreation.md)** and follow the steps using **[`_tutors/session04_tutor.xml`](../_tutors/session04_tutor.xml)** as the project's instructions. It takes about five minutes.

The tutor knows you have finished Sessions 01–03 and that you are working on the wine-quality drivers brief this week. It is configured to make callbacks to Sessions 01–03 where they help. It also knows about its siblings: ask it Python questions and it will redirect you to `python_stack_tutor`; ask it PowerShell questions and it will redirect you to `windows_powershell_tutor`.

Some example questions you might paste into the tutor while working through this notebook:

- *"I have $r = 0.48$ between alcohol and quality. Does that mean alcohol causes higher quality scores?"*
- *"What does it mean when a regression coefficient changes sign once I add another predictor?"*
- *"My $R^2$ is 0.34. Is that good? Is that bad?"*
- *"How is a partial correlation different from a regular correlation, in plain English?"*


***
## 4.3 Setup — imports and loading the data

Same opening move as the previous three weeks: imports first, data second, sanity-check third.

> **Callback to §1.3, §2.3, and §3.3.** Four libraries should already be old friends by now: `pandas`, `numpy`, `matplotlib`, and `seaborn`. This week we add two new arrivals: **`statsmodels`**, which gives us the full inferential-statistics machinery for linear regression (coefficient standard errors, $t$-tests, $p$-values, confidence intervals, diagnostics), and **`scikit-learn`** — the standard Python machine-learning library — from which we will use a couple of small pieces (`LinearRegression` for a sklearn-style fit alongside the statsmodels one). We will also keep `scipy.stats` from Session 03 for Pearson correlations with $p$-values, and `anthropic` for the API section at the end.


In [ ]:
# Import pandas under the alias pd; pandas gives us the DataFrame (tables of data).
import pandas as pd

# Import numpy under the alias np; numerical arrays, random generators, linear algebra.
import numpy as np

# Import matplotlib's plotting module under the alias plt.
import matplotlib.pyplot as plt

# Import seaborn under the alias sns; layered statistical plots.
import seaborn as sns

# Import the `stats` submodule of SciPy; this is where Pearson correlations (with p-values) live.
from scipy import stats

# Import statsmodels' formula API entry-point. We will use the .OLS class for ordinary least squares.
import statsmodels.api as sm

# VIF (variance inflation factor) lives inside statsmodels.stats.outliers_influence.
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Scikit-learn's LinearRegression — same fit, different API; we use both to make the connection visible.
from sklearn.linear_model import LinearRegression

# Jupyter magic that tells the notebook to display plots inline (directly in the notebook).
%matplotlib inline

# Cosmetic: clean seaborn default style.
sns.set_theme(style="whitegrid")

# Print a confirmation so we know the imports succeeded.
print("Libraries imported successfully.")


### Load the wine data

We will use the **UCI Wine Quality (red wine)** dataset — 1,599 Portuguese red wines, each with eleven physicochemical measurements taken during fermentation and a sensory quality score (0–10) assigned by tasting panellists. The dataset is released under **CC BY 4.0**, the same permissive licence as the UCI Bike Sharing data we used in Session 01.

One small twist: the UCI file uses **`;`** as its column separator, not a comma. `pandas` defaults to comma, so we need to tell it `sep=";"`. This is a tiny but common source of *"my DataFrame has one giant column"* surprises in practice.


In [ ]:
# Read the red-wine CSV into a DataFrame. The UCI file uses `;` as separator.
df = pd.read_csv(
    "../_data/wine_quality/winequality-red.csv",
    sep=";"
)

# Confirm what we have.
print(f"Loaded {len(df):,} wines, {df.shape[1]} columns.")


### Four checks before trusting the data

> **Callback to §1.4, §2.4, §3.4.** Same four checks, every session: shape, head, dtypes, missing values. Trust the data before trusting the analysis.


In [ ]:
# Check 1 — shape. (rows, columns).
df.shape


You should see `(1599, 12)` — **1,599 wines** and **12 columns** (eleven physicochemical predictors plus the `quality` score).


In [ ]:
# Check 2 — head. What does a row actually look like?
df.head()


In [ ]:
# Check 3 — dtypes. Every physicochemical column is float; `quality` is int (3 to 8 in this sample).
df.dtypes


In [ ]:
# Check 4 — any missing values?
df.isna().sum().sum()


You should see **0 missing values** across the entire DataFrame. UCI's wine quality file is unusually clean for a real-world dataset — we will not have to do any imputation today. (Imputation strategies are a topic for a later session; remember the data-limitations discipline from §2.12: even *"clean"* datasets carry implicit decisions made before we got the file.)

> **Mini-recap of §4.3.** Imports done; red-wine CSV loaded (1,599 rows × 12 columns, no missing values); four-check discipline applied. Every subsequent computation is on this DataFrame `df`.


***
## 4.4 Terminology — *dependent / independent / criterion / predictor / outcome / feature / endogenous / exogenous*

Before we go any further, we need a small terminology detour. The same two roles in a regression are called by **at least four different pairs of names**, depending on which scientific tradition wrote the textbook on the analyst's bookshelf. They all refer to the same thing. Not knowing this is the single most common reason participants get confused reading a paper, a tutorial, or a stakeholder email.

There are **two roles** in a drivers analysis:

1. **The variable we are trying to explain or predict.** In our case: the panel `quality` score.
2. **The variable (or variables) we are using to explain or predict it.** In our case: `alcohol`, `volatile acidity`, `sulphates`, and the rest.

Here are the names you will meet in the wild:

| Role | *Classical statistics* | *Experimental psychology / education* | *Machine learning* | *Econometrics* |
|---|---|---|---|---|
| The thing being explained (role 1) | **dependent variable** | **criterion** | **target**, **label**, **outcome**, **y** | **endogenous variable** |
| The thing doing the explaining (role 2) | **independent variable** | **predictor** | **feature**, **input**, **covariate**, **x** | **exogenous variable** |

They are all synonyms for the same two roles. Use whichever your stakeholder uses — but recognise the others when you see them. A few notes:

- **"Independent variable"** is the most widely-used term in classical statistics, and most often used in textbooks. It is also slightly misleading: in an *observational* dataset like ours the predictors are usually **not** independent of each other (alcohol and density are highly correlated in wine, for example). The word *independent* really means *"this is the side of the equation we are conditioning on"* — not *"these variables are statistically independent of each other"*. Multiple regression has to deal with that confusion explicitly (we will see VIF in §4.14).
- **"Endogenous" and "exogenous"** come from econometrics, where they carry an extra precise meaning beyond just *"left-hand side / right-hand side"*. An *exogenous* variable in the strict econometric sense is one that is determined **outside** the system being modelled — not affected by the outcome we are trying to explain. *Endogenous*, conversely, means determined **inside** the system, potentially affected by the outcome (or sharing common causes with it). In a basic regression these strict meanings often get loosened to be plain synonyms for *dependent / independent*, but the careful econometrician will distinguish.
- **"Extraneous variable"** is a less-standard term you may see in older experimental-design textbooks — it usually refers to a *nuisance* or *control* variable: something you measure not because it is interesting in itself, but because you want to *remove* its influence from the relationship of interest. The partial-correlation machinery in §4.12 is exactly the tool for handling extraneous variables.
- **"Features" and "target"** are the machine-learning vocabulary. A *feature* is just a predictor, a *target* is just an outcome. The ML literature also uses **"$X$"** (capital, often a matrix) for *all the predictors stacked together* and **"$y$"** (lowercase, a vector) for the target.

For the rest of this notebook we will mostly use the classical-statistics pair (**dependent / independent**, or sometimes **outcome / predictor** for readability) — and the matching mathematical symbols $y$ (outcome) and $x$ (predictor). When we get to multiple regression we will switch to $x_1, x_2, \ldots, x_p$ for several predictors at once.

> **Why this matters for your career.** The vocabulary you read in a paper or hear in a meeting tells you something about the speaker's training: a classical biostatistician will say *dependent variable*; a behavioural psychologist will say *criterion*; an economist may say *endogenous*; a machine-learning engineer will say *target* or *label*. They are all describing the same role. Hearing them all as synonyms is part of being fluent in the modern analyst landscape.


***
## 4.5 First look — visual exploration in multi-panel plots

> **Callback to §1.6 and §2.5.** Same discipline as the bike-rental data and the STATS19 file: look at the data before reaching for any formula. We will use **multi-panel** plots — several small plots sharing one figure — rather than scrolling past a dozen separately-rendered histograms. Multi-panel is the default in serious EDA precisely because the eye can compare distributions only when they sit next to each other.

We will produce three pictures:

1. A **4 × 3 grid of histograms**, one per column of the DataFrame — eleven predictors plus the `quality` score.
2. A **pairs plot** of a small, tractable subset of the most promising-looking variables, so we can see *all* their pairwise scatterplots at once.
3. A **single focused scatter** of `alcohol` vs `quality` with a best-fit line, as a teaser for the regression machinery we are about to build.


### 1. Histograms of every column

A histogram tells us the **shape** of each variable's distribution. We expect the physicochemical predictors to be roughly continuous and a mix of symmetric and right-skewed shapes; we expect `quality` to be a small set of integers.


In [ ]:
# Create a 4 x 3 grid of subplots, big enough to show 12 histograms comfortably.
fig, axes = plt.subplots(4, 3, figsize=(13, 12))

# Flatten the 4 x 3 array of axes so we can iterate linearly.
axes_flat = axes.flatten()

# Loop over every column in the DataFrame.
for ax, col in zip(axes_flat, df.columns):
    # Draw the histogram for this column on its subplot.
    ax.hist(df[col], bins=30, edgecolor="black")
    # Subplot title is the column name.
    ax.set_title(col, fontsize=10)
    # Lighter axis labels — column name is in the title already.
    ax.set_xlabel("")
    ax.set_ylabel("count", fontsize=8)

# Tidy spacing.
plt.tight_layout()
plt.show()


**What we see.** Most physicochemical variables are continuous and unimodal, several with a right tail (`residual sugar`, `chlorides`, `total sulfur dioxide` — typical fermentation chemistry). `pH` is the most symmetric, almost bell-shaped. `quality` is a small set of integers: most wines score 5 or 6, fewer score 4 or 7, and the extremes (3 and 8) are rare. That last point matters: our outcome is **ordinal-with-integer-values**, not continuous. Linear regression of `quality` on the predictors will still work as a useful descriptive tool, but the integer-grid character of the outcome will show up later in the residual plots.


### 2. Pairs plot of a tractable subset

The full 12-variable pairs plot would be 144 little panels — unreadable. We pick four predictors that, before any computation, look interesting (`alcohol`, `volatile acidity`, `sulphates`, `pH`) plus the outcome `quality`, and let `seaborn.pairplot` draw every pairwise scatter and every per-variable histogram in one figure.


In [ ]:
# Pick a tractable subset of variables: four candidate drivers plus quality.
subset = ["alcohol", "volatile acidity", "sulphates", "pH", "quality"]

# seaborn's pairplot draws a scatter for every pair off the diagonal, and a histogram on the diagonal.
sns.pairplot(df[subset], height=2.0, plot_kws={"alpha": 0.3, "s": 12})

# Render.
plt.show()


**What we see.** Two things jump out before we compute anything:

- The `alcohol` vs `quality` scatter (bottom-left or top-right depending on order) clearly *trends upward* — wines with more alcohol tend to receive higher panel scores. This will be our worked-example simple regression in §4.9.
- The `volatile acidity` vs `quality` scatter trends *downward* — higher acidity hurts the panel score. A second strong candidate driver.

Several other panels look noisier or flatter — the cloud has no obvious direction. We will quantify all of this in §4.6 (covariance) and §4.7 (correlation), and the ranking in §4.8 will confirm what the eye already suggests.


### 3. A focused scatter — `alcohol` vs `quality` with a best-fit line

One more picture before we touch formulas: the headline scatter, with a regression line drawn on top.


In [ ]:
# Single-panel scatter of alcohol (x) vs quality (y), with seaborn's regression line overlay.
# `quality` is an integer 3..8, so without help the points would compress into six perfectly
# horizontal lines of dots. The `y_jitter=0.25` argument adds a small random vertical wiggle
# to each dot FOR DISPLAY ONLY — the regression line is still fitted on the actual, un-jittered data.
plt.figure(figsize=(9, 5))

sns.regplot(
    x="alcohol",
    y="quality",
    data=df,
    y_jitter=0.25,                                          # display jitter — fit is on raw data
    scatter_kws={"alpha": 0.25, "s": 18, "color": "black"},
    line_kws={"color": "red"},
)

# Axis labels and title — flag the jitter in the title so the reader is not surprised.
plt.xlabel("alcohol (% by volume)")
plt.ylabel("quality (panel score, 0–10; jittered for display)")
plt.title("Red wine: alcohol vs panel quality (n = 1,599)")
plt.tight_layout()
plt.show()


**What we see.** A real, positive, but imperfect relationship.

**An important display note about the y-axis.** The panel `quality` score is an **integer** on a 3–8 grid — it is the same kind of variable as *"the count of stars in a movie review"*: discrete, ordered, with only a handful of distinct values. Without help, all 1,599 dots would compress into six perfectly horizontal lines of dots — one per integer score — and we would not be able to see how *many* wines sit at each combination. The `y_jitter=0.25` argument we passed to `regplot` adds a tiny random vertical wiggle to each dot **for display purposes only**; the red regression line is fitted on the actual, un-jittered data. With the jitter, the **density** of dots within each horizontal band becomes visible — and you can see how the dot density shifts rightward (toward higher alcohol values) as we move up to the higher integer quality scores.

The red line rises clearly from left to right, but the cloud of points around it is wide — the line is not the whole story. A perfectly deterministic relationship would have *all* points on the line. The vertical scatter around the line is the **residual** noise: the part of `quality` that `alcohol` alone does not explain.

We will return to *exactly this kind of picture* — but with a **continuous outcome** instead of an integer one — in §4.10, where the residual visual is the conceptual primitive on which the rest of the session is built. The integer character of `quality` will come back, honestly acknowledged, from §4.12 onward.

> **Mini-recap of §4.5.** Eleven predictors, one outcome. Visually, **alcohol** and **volatile acidity** look like the strongest candidate drivers; most other predictors look noisier. Time to put numbers on the eyeballing.


***
## 4.6 Covariance — measuring how two variables move together

We have just *eyeballed* a positive relationship between `alcohol` and `quality`. *"Eyeballed"* is not enough for the winemaker — she wants a number. The first number an analyst reaches for, when asked *"do these two variables move together?"*, is the **covariance**.

### The idea, in one sentence

> *"Take every pair of points. For each point, ask: is its $x$ above or below the $x$ average? And is its $y$ above or below the $y$ average? Multiply those two signed deviations together, then average."*

If, **on average across the dataset**, the wines with above-average alcohol also have above-average quality (and below-average alcohol goes with below-average quality), then most of those products will be positive, the average will come out positive, and we will say the two variables **co-vary positively**. If the pattern is the reverse — higher alcohol pairs with lower quality — the products will mostly be negative and the covariance will be negative. If there is no systematic relationship, positives and negatives will cancel out and the covariance will be near zero.

### The formula

For two variables $X$ and $Y$ measured on the same $n$ observations:

$$\mathrm{cov}(X, Y) \;=\; \frac{1}{n - 1} \sum_{i=1}^{n} \bigl(x_i - \bar{x}\bigr)\bigl(y_i - \bar{y}\bigr)$$

Reading every symbol:

- $X$ and $Y$ — the two variables (each is a column of $n$ numbers).
- $x_i$ and $y_i$ — the $i$-th observation of each (e.g., the $i$-th wine's alcohol and quality).
- $\bar{x}$ and $\bar{y}$ — the **sample means** of $X$ and $Y$. (We met sample means in Session 01 §1.5.1.)
- $x_i - \bar{x}$ — the **deviation** of the $i$-th observation from the $x$ mean. Positive if above average, negative if below.
- $(x_i - \bar{x})(y_i - \bar{y})$ — the **product of the two signed deviations** for the $i$-th observation. Positive when both are on the same side of their means; negative when they are on opposite sides.
- $\sum_{i=1}^{n}$ — sum that product over all $n$ observations.
- $\frac{1}{n - 1}$ — divide by $n - 1$, the same **Bessel's correction** we met for the sample variance in Session 01 §1.5.4. (When $X = Y$, the covariance formula becomes the variance formula — covariance generalises variance to two variables.)

### A tiny worked example by hand

Suppose we have only four wines with `(alcohol, quality)` pairs:

| wine | alcohol | quality |
|---:|---:|---:|
| 1 | 9 | 4 |
| 2 | 10 | 5 |
| 3 | 11 | 6 |
| 4 | 12 | 7 |

Means: $\bar{x} = 10.5$, $\bar{y} = 5.5$. Deviations:

| wine | $x_i - \bar{x}$ | $y_i - \bar{y}$ | product |
|---:|---:|---:|---:|
| 1 | $-1.5$ | $-1.5$ | $+2.25$ |
| 2 | $-0.5$ | $-0.5$ | $+0.25$ |
| 3 | $+0.5$ | $+0.5$ | $+0.25$ |
| 4 | $+1.5$ | $+1.5$ | $+2.25$ |

Sum of products: $5.0$. Divide by $n - 1 = 3$: covariance $\approx 1.67$. Positive, large relative to the units involved — every wine with above-average alcohol also has above-average quality, by exactly the same amount. (This is the maximally-aligned case.)


### Compute covariance on the real wine data


In [ ]:
# Pull alcohol and quality out as plain numpy arrays.
alcohol = df["alcohol"].values
quality = df["quality"].values

# Sample size.
n = len(alcohol)

# Sample means.
alcohol_mean = alcohol.mean()
quality_mean = quality.mean()

# Sample deviations from the mean, element-wise.
alcohol_dev = alcohol - alcohol_mean
quality_dev = quality - quality_mean

# Sum of products of deviations.
sum_of_products = (alcohol_dev * quality_dev).sum()

# Sample covariance: sum of products divided by (n - 1).
cov_by_hand = sum_of_products / (n - 1)

# Print the by-hand result.
print(f"n                            = {n}")
print(f"mean alcohol                 = {alcohol_mean:.4f} % by volume")
print(f"mean quality                 = {quality_mean:.4f}")
print(f"cov(alcohol, quality) by hand = {cov_by_hand:.6f}")


### Verify against numpy


In [ ]:
# np.cov returns a 2x2 covariance matrix; we want the off-diagonal entry.
# ddof=1 selects Bessel's correction (n-1 in the denominator), matching our hand computation.
cov_matrix = np.cov(alcohol, quality, ddof=1)
print("Covariance matrix from np.cov (ddof=1):")
print(cov_matrix)
print()
print(f"Off-diagonal entry        = {cov_matrix[0, 1]:.6f}")
print(f"By-hand computation above = {cov_by_hand:.6f}")


You should see both numbers agree exactly: **$\mathrm{cov}(\text{alcohol}, \text{quality}) \approx 0.4098$**. Positive, as the §4.5 scatter suggested.

### So what?

The covariance is **positive**, which means *"alcohol and quality move together on average"*. Good — that confirms the visual. But the number itself, *0.4098*, is hard to interpret. Its **units** are *"percent-alcohol × quality-point"* — a unit nobody knows how to feel about. Worse, if we had measured alcohol in different units (parts per thousand, or grams per litre), the covariance would change by exactly that scale factor, even though the *underlying relationship* would be identical.

That is a real practical problem. We cannot compare *cov(alcohol, quality)* to *cov(volatile acidity, quality)* directly, because the predictors live on different scales. **We need to rescale.** That is exactly what correlation is for, and it is the next section.

> **Mini-recap of §4.6.** Covariance is the average of *signed deviation products*. It is positive when the two variables move together, negative when they move oppositely, and near zero when there is no systematic relationship. **But it is in inconvenient units that depend on the units of $X$ and $Y$ — so it is not directly comparable across variable pairs.**


***
## 4.7 Correlation — covariance, rescaled to $[-1, +1]$

The problem with covariance is **units**. The fix is to **standardise both variables before taking the covariance**. Once each variable is on a *"how many of its own standard deviations away from its own mean is each value"* scale, the units cancel out and the resulting number lives on a fixed, interpretable range: $-1$ to $+1$.

That rescaled covariance has a famous name: it is the **Pearson coefficient of linear correlation**, almost always written $r$.

### Standardisation — the $z$-score

> **Callback to §1.5 and §1.10.** We have already met *"standardise"* in spirit — that is what the standard error did for the sample mean. Now we apply the same trick *element by element* to a column.

The **$z$-score** of an observation $x_i$ is:

$$z(x_i) \;=\; \frac{x_i - \bar{x}}{s_X}$$

Reading every symbol:

- $x_i$ — the $i$-th observation of $X$.
- $\bar{x}$ — the sample mean.
- $s_X$ — the sample standard deviation. (Session 01 §1.5.4.)
- $z(x_i)$ — the $i$-th observation expressed in **units of standard deviations from the mean**. A $z$-score of $+1.5$ means *"this value is 1.5 standard deviations above the mean"*. A $z$-score of $-0.4$ means *"0.4 standard deviations below the mean"*.

A $z$-scored variable has, by construction, **mean zero and standard deviation one**. That is what *"standardised"* means.

### Pearson correlation = covariance of $z$-scores

Here is the punchline, in one line. The Pearson correlation $r_{XY}$ between $X$ and $Y$ is **exactly the covariance of their $z$-scored versions**:

$$r_{XY} \;=\; \mathrm{cov}\bigl(z(X),\, z(Y)\bigr)$$

If you understand standardisation and you understand covariance, you understand correlation. There is no third concept hiding inside; correlation is just *"the covariance you get after putting both variables on the same standardised scale"*.

The textbook also writes the same quantity in unstandardised form:

$$r_{XY} \;=\; \frac{\mathrm{cov}(X, Y)}{s_X \cdot s_Y}$$

These two formulas are **algebraically identical**. The first one tells you *what correlation is* (a covariance of standardised variables); the second one tells you *how to compute it given the raw covariance and the two standard deviations*. Use whichever is easier in context.


### Compute correlation on the wine data — both ways


In [ ]:
# Method 1: standardise both variables, then take their covariance.
# Compute each variable's sample standard deviation (ddof=1 = Bessel's correction).
alcohol_sd = alcohol.std(ddof=1)
quality_sd = quality.std(ddof=1)

# Standardise each variable: subtract the mean, divide by SD. Element-wise on the numpy arrays.
z_alcohol = (alcohol - alcohol_mean) / alcohol_sd
z_quality = (quality - quality_mean) / quality_sd

# Covariance of the standardised variables.
r_from_zcov = np.cov(z_alcohol, z_quality, ddof=1)[0, 1]

# Method 2: textbook formula, using the raw covariance we already computed.
r_from_textbook = cov_by_hand / (alcohol_sd * quality_sd)

# Method 3: scipy.stats.pearsonr, which also returns a two-sided p-value for the null r = 0.
r_scipy, p_value = stats.pearsonr(alcohol, quality)

# Print all three to verify they agree.
print(f"r from cov of z-scores       = {r_from_zcov:.6f}")
print(f"r from textbook cov/(sx*sy)  = {r_from_textbook:.6f}")
print(f"r from scipy.stats.pearsonr  = {r_scipy:.6f}")
print()
print(f"p-value (two-sided test of H0: rho = 0) = {p_value:.3e}")


You should see all three values agree at $r \approx 0.4762$. That is the **Pearson correlation between alcohol and quality**: a moderate positive linear relationship. The $p$-value is astronomically small ($\sim 10^{-91}$), so under the null hypothesis *"the two variables are uncorrelated in the underlying population"*, observing $|r|$ this large in a sample of 1,599 would be essentially impossible — we reject that null with enormous margin.

### How to read a correlation

The correlation $r$ lives on a fixed scale from $-1$ to $+1$, and the shape of the cloud changes systematically as you move along that scale:

- $r = +1$: a perfect ascending straight line. Knowing $X$ tells you $Y$ exactly.
- $r \approx +0.7$ to $+0.9$: a clear upward trend, with modest scatter.
- $r \approx +0.3$ to $+0.5$: a visible upward trend, but the cloud is wide — our `alcohol`/`quality` case.
- $r \approx 0$: no linear association; the cloud is round or otherwise structureless along a straight line.
- $r \approx -0.3$ to $-0.9$: same as above, in mirror image — a downward trend.
- $r = -1$: a perfect descending straight line.

### One enormous caveat — *$r = 0$ does not mean "no relationship"*

The word **linear** in *"Pearson coefficient of **linear** correlation"* matters. Pearson $r$ measures one specific shape of association: a straight-line one. A perfectly clear **non-linear** relationship can give $r \approx 0$.


In [ ]:
# Build a textbook demo: y = x^2 with a sprinkle of noise.
# Create a numpy random generator with a fixed seed so the demo is reproducible.
rng = np.random.default_rng(42)

# x ranges from -3 to +3.
x_demo = np.linspace(-3, 3, 200)

# y is x squared plus a small Normal noise term.
y_demo = x_demo**2 + rng.normal(scale=0.2, size=200)

# Pearson r between the two.
r_demo, p_demo = stats.pearsonr(x_demo, y_demo)

# Plot the scatter so we can see the relationship the eye sees easily.
plt.figure(figsize=(7, 4))
plt.scatter(x_demo, y_demo, alpha=0.6, s=18, color="black")
plt.title(f"y = x² + noise   —   Pearson r = {r_demo:+.4f}   (essentially zero)")
plt.xlabel("x")
plt.ylabel("y")
plt.tight_layout()
plt.show()


The picture screams *"clear relationship!"* — every value of $x$ has a *very* predictable $y$ — and yet Pearson $r$ is essentially zero. Why? Because for every value of $x$ above zero there is a *matching* value of $x$ below zero with the same $y$. The signed deviations $(x_i - \bar{x})$ cancel out in the covariance sum. Pearson $r$ has not failed at its job — it has succeeded at *the specific job it was designed for*. That job is *"measure the strength of a **linear** association"*. For non-linear shapes you need other tools (Spearman rank correlation, distance correlation, mutual information, or — most often — visualising the data first and choosing an appropriate model second).

The practical lesson: **never report only $r$.** Always look at the scatter.

> **Mini-recap of §4.7.** Pearson $r$ is the covariance of the $z$-scored variables — a unitless, scale-invariant measure of *linear* association on $[-1, +1]$. For `alcohol` vs `quality` on red wine: $r \approx 0.476$, with a $p$-value essentially zero. **But $r$ only sees straight lines.** Always look at the scatter.


***
## 4.8 Correlation matrix and ranking the candidate drivers

We have $r$ for one predictor (`alcohol`). The winemaker asked about **all eleven**. Doing the computation pair by pair would be tedious; `pandas` does the whole correlation matrix in one call.

A correlation matrix of $p$ variables is a $p \times p$ table where entry $(i, j)$ is the Pearson correlation between variable $i$ and variable $j$. The diagonal is always $1$ (every variable correlates perfectly with itself), and the matrix is symmetric (the correlation of $X$ with $Y$ equals the correlation of $Y$ with $X$).


In [ ]:
# Compute the full 12 x 12 correlation matrix of every column with every column.
corr = df.corr(method="pearson")

# Show it, rounded for readability.
corr.round(3)


That is a lot of numbers. Two visualisations make it easier to read:

1. A **heatmap** of the full matrix — colour replaces digits, so the eye can scan.
2. A **bar chart of just the row that matters for our brief** — the correlations of each predictor with `quality`, ranked by absolute size.


In [ ]:
# Side-by-side: heatmap of the full matrix on the left, ranked bar chart on the right.
fig, axes = plt.subplots(1, 2, figsize=(15, 6), gridspec_kw={"width_ratios": [1.2, 1]})

# Left subplot: heatmap.
# annot=True prints the numeric correlation in each cell. fmt=".2f" rounds to 2 decimals.
# center=0 puts the colour gradient's midpoint at r=0; coolwarm goes blue-low to red-high.
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    cbar_kws={"shrink": 0.6},
    ax=axes[0],
)
axes[0].set_title("Pearson correlation matrix (all 12 variables)")

# Right subplot: just the correlations with quality, ranked by absolute value.
# Pull the quality column from the matrix, drop the quality–quality 1.0 entry, sort by |r|.
corr_quality = corr["quality"].drop("quality").sort_values(key=abs, ascending=True)

# Horizontal bar chart so the names are readable on the y-axis.
colors = ["#d62728" if v < 0 else "#1f77b4" for v in corr_quality.values]
axes[1].barh(corr_quality.index, corr_quality.values, color=colors, edgecolor="black")
axes[1].axvline(0, color="black", linewidth=0.7)
axes[1].set_xlim(-0.5, 0.55)
axes[1].set_xlabel("Pearson r with quality")
axes[1].set_title("Candidate drivers, ranked by |r|")

plt.tight_layout()
plt.show()


**What we see.**

- The strongest linear association with `quality` is **alcohol** ($r \approx +0.476$) — positive, as we already knew.
- The strongest *negative* association is **volatile acidity** ($r \approx -0.391$) — wines with higher volatile-acidity scores get lower panel scores. (This matches winemaker intuition: volatile acidity is the chemistry behind a "vinegary" off-note.)
- A second tier of candidates: **sulphates** ($+0.251$), **citric acid** ($+0.226$), **total sulfur dioxide** ($-0.185$), **density** ($-0.175$).
- A third tier so weak it is essentially noise: **pH** ($-0.058$), **free sulfur dioxide** ($-0.051$), **residual sugar** ($+0.014$).

This ranking is our **first concrete answer to the winemaker's question**: a list of candidate drivers, ranked by strength of linear association, with the sign of each association explicit. But — and this is critical — a correlation of $0.476$ does *not* mean *"alcohol explains 47.6% of quality"*. To answer the *"how much"* question precisely we need to upgrade from correlation to **regression**, and to formal *variance accounting*. That is the next three sections.

> **Mini-recap of §4.8.** The Pearson correlation matrix lets us rank candidate drivers by strength of linear association. For red wine: **alcohol (+)** and **volatile acidity (−)** are the strongest single drivers; **sulphates (+)** and **citric acid (+)** are next; **pH, free sulfur dioxide, residual sugar** look like noise. We will revisit this ranking with regression in §4.9 and §4.14.


***
## 4.9 Simple linear regression — the model

Correlation tells us *whether* and *how strongly* two variables move together. It does **not** answer the next two practical questions a stakeholder will always ask:

1. *"If predictor $X$ went up by one of its units, by how much would outcome $Y$ move?"* — the **size** of the effect, in the original units.
2. *"How well does a straight line through this cloud actually fit?"* — the **goodness of fit**.

For both, we need the **simple linear regression model**.

### A pedagogical detour — why we are *not* using `quality` as the outcome in §§4.9–4.11

Before we write the model down, one practical decision. The brief is about `quality`, and we will fully answer the brief in §§4.12–4.16. But for the next three sections — where we **introduce** the simple regression machinery, the residual scatterplot (§4.10), and $R^2$ (§4.11) — we are going to use a **different worked example**: `density ~ alcohol`.

Why? Because `quality` is an **integer** on a 3–8 grid. As we saw in §4.5, that turns any scatter involving `quality` into a stack of six horizontal bands. For an introductory residual visual, those bands fight the picture: every residual snaps to a near-integer length, the *"cloud around the line"* intuition is hidden, and a beginner has to mentally subtract the integer-grid artifact before reading the figure. **It is much easier to learn what a residual *is* on two continuous variables**, with a clean cloud and dashed segments of every possible length.

We will use **`density`** as the outcome and **`alcohol`** as the predictor. Both are continuous physical measurements:

- `alcohol` is the % alcohol by volume (typical range ~8.4 to 14.9 in our dataset).
- `density` is the wine's mass per unit volume in g/cm³ (typical range ~0.990 to 1.004).

Chemically the relationship is intuitive: pure ethanol is less dense than water, so more alcohol in a wine pulls the density downward. We expect a **negative slope** — and as we already saw in §4.8, $r(\text{alcohol}, \text{density}) \approx -0.496$. **The same machinery we build here on `density ~ alcohol` is exactly the machinery we apply to `quality` in §4.12 and beyond.** Nothing changes about the formulas; only the picture gets cleaner while we are first learning.

### The model

We posit that the outcome $y$ is generated, for each observation, as a straight-line function of the predictor $x$ plus some random noise:

$$y_i \;=\; \beta_0 \;+\; \beta_1 \, x_i \;+\; \varepsilon_i$$

Reading every symbol:

- $y_i$ — the $i$-th observed value of the outcome (the wine's `density`, in our worked example).
- $x_i$ — the $i$-th observed value of the predictor (the wine's `alcohol`).
- $\beta_0$ — *"beta-zero"*, the **intercept**: the model's predicted value of $y$ when $x = 0$.
- $\beta_1$ — *"beta-one"*, the **slope**: the **change in $y$ associated with a one-unit increase in $x$**. This is the number we will hand back to the winemaker (whenever the regression is meaningful — for our pedagogical fit, the *winemaker* does not really care about how alcohol changes density; she cares about how things change `quality`, which is §4.12 onward).
- $\varepsilon_i$ — *"epsilon-sub-i"*, the **error term** for the $i$-th observation: the part of $y_i$ that the straight-line model **does not** capture. Some observations land above the line ($\varepsilon_i > 0$), some below ($\varepsilon_i < 0$).
- The Greek letters $\beta_0$ and $\beta_1$ are **population parameters** — they describe the true line through the underlying process. From the sample we will compute **estimates** $\hat{\beta}_0$ and $\hat{\beta}_1$ (with hats).

### How the line is chosen — *ordinary least squares*

There are infinitely many lines we could draw through a cloud of points. The model needs a rule for picking one. The rule everyone agrees on — for reasons that go back to Gauss in 1805 — is:

> *"Pick the line that makes the sum of the **squared** vertical distances from each point to the line as small as possible."*

Those vertical distances have a name we will see again and again: **residuals**. Squaring them serves two purposes: (i) it stops positive and negative residuals from cancelling, and (ii) it penalises large residuals more than small ones, so the chosen line is the one that *especially* avoids being far from any single point.

This rule is **ordinary least squares (OLS)**. It has a closed-form solution — there is a formula for $\hat{\beta}_0$ and $\hat{\beta}_1$ — but for now we will let Python compute the estimates for us, twice, in two different libraries.


### Fit the model — `scikit-learn` and `statsmodels`

Python has two standard ways to fit a linear regression: **scikit-learn** (the machine-learning library — terse, fast, fits-then-predicts) and **statsmodels** (the classical-statistics library — richer summary tables, $p$-values, confidence intervals, diagnostic machinery). We will fit the same model both ways, so you can see they agree and recognise the syntax of each.


In [ ]:
# --- Fit 1: scikit-learn ---
# Pull density out as a numpy array; we will treat it as the outcome y in our worked example.
density = df["density"].values

# The sklearn API expects X as a 2D array (matrix) and y as a 1D array (vector), even for one predictor.
# np.reshape(-1, 1) turns the 1D alcohol array into an n-by-1 column matrix.
X_sk = alcohol.reshape(-1, 1)
y_sk = density

# Create the regression object and fit it on the data.
sk_model = LinearRegression()
sk_model.fit(X_sk, y_sk)

# Pull out the intercept and the single slope.
sk_intercept = sk_model.intercept_
sk_slope     = sk_model.coef_[0]

# Print sklearn's estimates.
print("scikit-learn LinearRegression  (density ~ alcohol):")
print(f"  intercept (beta_0_hat) = {sk_intercept:.6f}")
print(f"  slope     (beta_1_hat) = {sk_slope:+.6f}    (g/cm³ per +1% alcohol)")


In [ ]:
# --- Fit 2: statsmodels ---
# statsmodels needs us to add an explicit constant column (a column of 1s) for the intercept.
# sm.add_constant prepends that column to a copy of the predictor matrix.
X_sm = sm.add_constant(alcohol)

# OLS() builds the model object; .fit() actually runs the fit.
sm_model = sm.OLS(density, X_sm).fit()

# Print statsmodels' rich summary.
print(sm_model.summary())


### Reading the summary

Three things to lock in from the output.

**1. The estimated coefficients.**

- $\hat{\beta}_0$ (the intercept, labelled `const`) is approximately **+1.006**. Literally: *"a hypothetical wine with 0% alcohol would have a predicted density of 1.006 g/cm³"*. This is **not a useful claim on its own** — no wine has 0% alcohol, so the intercept here is extrapolating far outside the observed range. The intercept's job in a regression is mostly to make the *slope* interpretation correct; it is not always a quantity worth reporting to a stakeholder.
- $\hat{\beta}_1$ (the slope, labelled `x1`) is approximately **−0.000879 g/cm³ per +1% alcohol**. This **is** worth reporting: *"each one-percent increase in alcohol is associated, on average across this dataset, with a decrease in density of about 0.000879 g/cm³ — equivalently, about 0.88 milligrams per cubic centimetre, or about 0.88 g/L"*. For a wine going from 9% to 14% alcohol (a five-point alcohol swing), that adds up to about $5 \times 0.88 \approx 4.4$ g/L of density change — a physically meaningful amount.

**2. The standard error, $t$-statistic, and $p$-value of the slope.**

- The standard error of $\hat{\beta}_1$ is about **0.000038**, sometimes shown in scientific notation as `3.85e-05`. It is a small number, because we have 1,599 wines and the predictor varies meaningfully across them.
- The $t$-statistic is about **−22.84** — the slope is roughly 23 standard errors *below* zero. The next section unpacks what that statement literally means and what it is comparing.
- The $p$-value is essentially zero ($\sim 10^{-100}$). Under the null hypothesis $H_0: \beta_1 = 0$ (no linear association between alcohol and density in the underlying population), the chance of observing a sample slope this extreme is vanishingly small. We **reject** that null with enormous margin.

**3. $R^2 \approx 0.246$.** The line explains about **24.6% of the variance** in density. We will unpack what $R^2$ literally means in §4.11.


### What the t-test in this table is actually comparing

The `t` and `P>|t|` columns in the statsmodels summary are a **t-test** applied to a regression coefficient — and the test deserves its own paragraph, because the question it asks is *not literally the same* as the t-tests we met in Session 03, even though the machinery is identical.

> **Callback to Session 03 §3.5 and §3.7.** In Session 03 we used a t-test to compare **two group means** — *"is variant A's mean time-on-page different from variant B's?"*. Here we use the same statistical machinery — *"how many standard errors does our estimate sit away from a null value of zero?"* — but the **two things being compared** are different. Same recipe, different ingredients.

#### The two things being compared in a regression t-test

A t-test on a regression coefficient compares:

1. **Our observed slope estimate** $\hat{\beta}_1 \approx -0.000879$ — the number we computed from our 1,599-wine sample.
2. **A null-hypothesis value** $\beta_1^{\text{null}} = 0$ — the slope a *"no relationship"* horizontal line would have.

The test asks: *"if the true population slope were exactly zero — i.e., if alcohol had no linear association with density at all — how unlikely is it that we would have observed a sample slope as far from zero as $-0.000879$?"*

#### Picturing it — the parallel-universe story (again)

> **Callback to §1.9.** We used the *"1,000 parallel-universe analysts"* story before, to motivate the sampling distribution of the mean. The same story applies here, with the slope playing the role of the mean.

Imagine a parallel universe in which the true relationship between alcohol and density is **flat** — no linear association at all. In that universe, the *population* regression line is horizontal: predicted density does not change with alcohol. But every sample of 1,599 wines you draw from that universe will produce a slightly different *sample* slope $\hat{\beta}_1$ by random chance alone. The collection of all such sample slopes is the **sampling distribution of $\hat{\beta}_1$ under the null** — centred on zero (because the true slope is zero), with a standard deviation equal to the **standard error of the slope**, $\widehat{\mathrm{SE}}(\hat{\beta}_1)$.

> **Callback to §1.9 and §1.10.** *"The sampling distribution of $\hat{\beta}_1$"* is to *"the slope"* what *"the sampling distribution of the sample mean"* was to *"the mean"* — the distribution of estimates we would get across many parallel-universe re-samples. The **standard error of the slope** plays the same role the standard error of the mean did in Session 01: it is the **standard deviation of an estimator** across hypothetical re-samples.

#### The t-statistic — *"how many standard errors away from the null?"*

The t-statistic for a regression slope is the precise version of *"how many standard errors away from the null value is our observed estimate?"*:

$$t \;=\; \frac{\hat{\beta}_1 \;-\; \beta_1^{\text{null}}}{\widehat{\mathrm{SE}}(\hat{\beta}_1)} \;=\; \frac{\hat{\beta}_1 \;-\; 0}{\widehat{\mathrm{SE}}(\hat{\beta}_1)} \;=\; \frac{\hat{\beta}_1}{\widehat{\mathrm{SE}}(\hat{\beta}_1)}$$

Reading every symbol:

- $\hat{\beta}_1$ — our observed sample slope ($-0.000879$ here).
- $\beta_1^{\text{null}}$ — the value the test is comparing against, which is **almost always zero** because *"no relationship"* is the natural default position we want to falsify.
- $\widehat{\mathrm{SE}}(\hat{\beta}_1)$ — the standard error of the slope (the `std err` column in the table).

For our regression: $t = -0.000879 \,/\, 0.000038 \approx -22.84$. **The slope is roughly 23 standard errors below zero.** In the null universe where the true slope is exactly zero, observing a sample slope this far from zero is wildly implausible.

> **Aside on the sign.** The negative sign of $t$ here just says *"our estimate is **below** the null value"*, not *"above"*. For a two-sided test (which is the default), only $|t|$ matters for the p-value — a $t$ of $+22.84$ and a $t$ of $-22.84$ give the same p-value. The sign tells you the **direction** of the deviation from the null, which is informative for the memo: *"density goes **down** as alcohol goes up."*

#### The p-value — the tail area under the null distribution

Under the null hypothesis $H_0: \beta_1 = 0$, the statistic $t$ follows a **t-distribution with $n - 2 = 1{,}597$ degrees of freedom** (one observation per data point, minus the two parameters $\beta_0$ and $\beta_1$ we had to estimate to fit the line). For sample sizes this large, the t-distribution is essentially indistinguishable from a standard Normal.

The **p-value** is the two-tailed area beyond $|t|$ in that distribution. For our slope, $p \approx 4 \times 10^{-100}$ — vanishingly small. The decision rule from Session 03 carries over unchanged:

> *"Reject $H_0$ at level $\alpha$ if $p < \alpha$."*

We **reject the null** that alcohol has no linear association with density, with enormous margin. In plain English: *"a flat, no-association line is essentially impossible given what we observed."*

#### Summary in one sentence

> **The t-test on a regression coefficient compares your observed slope estimate $\hat{\beta}_1$ to a null value of zero, in units of the slope's own standard error, and asks how plausible the null is given the data.** Big $|t|$ → small p-value → reject the null. Small $|t|$ → large p-value → fail to reject. Same logic as every t-test you have ever seen; the *"thing being tested"* is just a slope now instead of a difference of means.

We will see in §4.14 that the multiple regression's summary table has **one row per predictor**, and **each row is its own t-test** — with one subtle but important twist about what the null literally means once there is more than one predictor in the model.


### Word of caution — *association, not causation*

A negative slope of $-0.000879$ g/cm³ per +1% alcohol does **not** mean *"if we add alcohol to a finished wine, its density will drop by 0.000879 g/cm³"*. It means *"in this observational sample of 1,599 already-bottled wines, the ones with more alcohol also happened to be less dense"*. The cleanest causal interpretation (*"ethanol is less dense than water, so adding ethanol lowers density"*) is supported by basic chemistry — but the *regression* by itself does **not** establish that. Regression on observational data delivers **candidates for causation**, never **conclusions about causation**. This is a hill the careful analyst always dies on, and the §4.12 partial-correlation machinery will let us at least *probe* whether each association survives controlling for other variables.

> **Mini-recap of §4.9.** A simple linear regression of `density` on `alcohol` estimates a slope of $\hat{\beta}_1 \approx -0.000879$ g/cm³ per +1% alcohol — *"each extra percent of alcohol is associated with a density drop of about 0.88 g/L on average"* — with a standard error of $3.8 \times 10^{-5}$ and a $p$-value indistinguishable from zero. The fit explains about 24.6% of the variance in density. We used `density ~ alcohol` as the worked example because the picture (in §4.10) is cleanest with two continuous variables. We will apply the *same machinery* to `quality` from §4.12 onward.


***
## 4.10 The residual scatterplot — the single most important picture in this notebook

If you remember **one figure** from this session, it should be the one in this section. Almost everything that follows — $R^2$, partial correlation, part correlation, multiple regression's *"all else equal"* claim, the diagnostic plots — is built on the same visual primitive: a scatterplot in which each data point sits next to its model prediction, with the **residual** (the vertical gap between them) drawn as a short dashed segment.

Once you can *see* what a residual is, every later formula is just *"do something with these dashed segments"*. Without that picture, the formulas float free.

### What we are about to draw

We will plot the same `density` vs `alcohol` scatter the simple regression in §4.9 was built on, but with three layers stacked:

1. **The actual data points**: white-filled, black-edged dots. *What the wine actually had as its measured density.*
2. **The regression line**: in red. *What the model predicts for each `alcohol` value.*
3. **The predicted points and the residuals**: a small red dot directly below (or above) each white dot, at the model's prediction, connected by a **vertical dashed segment**. *The signed gap between the data and the model.*

We will draw the picture on a random subsample of **80 wines** (out of 1,599) — using all 1,599 dashed segments at once would smear into an unreadable mess. The cleanliness of the picture is the whole point.


In [ ]:
# Pick a manageable subset of the data for the residual visual: 80 wines drawn at random.
# (With all 1,599 points the dashed segments would overlap into a black mess.)
# Use a default_rng for reproducibility, the modern numpy convention.
rng = np.random.default_rng(42)
sample_idx = rng.choice(len(alcohol), size=80, replace=False)

# The same 80-wine subsample — but now we are plotting density on the y-axis (the continuous
# outcome from our §4.9 worked example), not the integer `quality`. The dashed segments will
# now take on every possible length, giving us the canonical "cloud around a line" picture.
x_plot = alcohol[sample_idx]
y_plot = density[sample_idx]
yhat_plot = sm_model.predict(sm.add_constant(x_plot))   # model predictions at those x values

# Create the figure.
plt.figure(figsize=(11, 6))

# Layer 1: the regression line, drawn across the full observed range of alcohol.
x_line = np.linspace(alcohol.min(), alcohol.max(), 200)
y_line = sk_intercept + sk_slope * x_line
plt.plot(x_line, y_line, color="red", linewidth=1.2, label="regression line")

# Layer 2: the vertical dashed residual segments, from each data point down (or up) to its prediction.
for xi, yi, yh in zip(x_plot, y_plot, yhat_plot):
    plt.plot([xi, xi], [yi, yh], color="black", linestyle="--", linewidth=0.7)

# Layer 3a: the actual data points — black ring, white fill.
plt.scatter(x_plot, y_plot, s=70, facecolor="white", edgecolor="black", linewidth=1.0,
            zorder=3, label="actual density (data)")

# Layer 3b: the predicted points — small red dots, sitting on the regression line.
plt.scatter(x_plot, yhat_plot, s=20, color="red", zorder=4, label="predicted density (model)")

# Titles and labels.
plt.title("Residuals visualised — density ~ alcohol, on 80 wines drawn at random")
plt.xlabel("alcohol (% by volume)")
plt.ylabel("density (g/cm³)")
plt.legend(loc="upper right")
plt.tight_layout()
plt.show()


### Reading the picture, in plain English

Look at any single white dot. It represents one real wine, with its measured alcohol on the $x$-axis and its measured density on the $y$-axis. Now follow the dashed segment from that white dot until it hits the red dot directly above or below it. That **red dot is what the model would have predicted for that wine, given only its alcohol**. The **dashed segment is the residual** — the signed gap between the wine's actual density and the model's prediction:

$$\text{residual}_i \;=\; y_i \;-\; \hat{y}_i$$

where $\hat{y}_i = \hat{\beta}_0 + \hat{\beta}_1 x_i$ is the prediction for the $i$-th wine.

Some white dots sit above their red companions — the model **under-predicted** that wine (the residual is positive). Some sit below — the model **over-predicted** (the residual is negative). A few sit almost exactly on the line — the model nailed it for that wine.

Notice that the dashed segments take on **every possible length** — short ones near the line, long ones for wines far from it, some upward and some downward. This is exactly what we *could not* see in §4.5 with `quality` on the y-axis, because there the integer outcome forced the dots into six horizontal bands and every residual snapped to a near-integer length. **Continuous outcome → continuous-length residuals → a true cloud around the line.** That is the picture you want anchored in your head as the visual primitive for everything that follows.

**The ordinary-least-squares line is, by definition, the unique straight line that makes the sum of the squared lengths of all those dashed segments as small as possible.** Move the line up, down, tilt it; the sum of squared dashed-segment lengths will only get larger. That is the geometric content of *"least squares"*.

### Why this picture is the unit of currency for the rest of the session

- In **§4.11** we will measure how big the dashed segments are *in total* relative to the original spread of $y$ — that ratio is $R^2$.
- In **§4.12** we will compute the residuals from regressing one variable on another, and then compute the correlation of *those residuals* with a third variable — that is partial correlation.
- In **§4.13** we will do the same trick, but residualise only one side — that is part correlation.
- In **§4.14** we will read a multiple regression's coefficients in exactly the same language: *"the slope of $y$ on the part of $x_k$ that is left over after removing what the other predictors already explained"*.
- In the diagnostic plots inside §4.14 we will plot the residuals on the $y$-axis and look for patterns; any structure left in the residuals is a hint that the model is missing something (with one important caveat about integer outcomes, which we will face honestly).

**Whenever you read the word "residual" from now on, picture a dashed vertical segment in this plot.** That is what it means.

> **Mini-recap of §4.10.** A residual is the signed vertical distance from a data point to its model prediction. The OLS line is the line that minimises the sum of squared residuals. We drew the picture on `density ~ alcohol` because two continuous variables give the cleanest illustration; the same concept applies — with the same dashed-segment interpretation — to any regression we fit, including the `quality` regressions to come.


***
## 4.11 $R^2$ — what the line explains, what it does not

We have a slope, an intercept, and a residual scatterplot. The natural follow-up question — and the one stakeholders ask first — is: *"how good is the fit?"* The textbook answer is a number called **$R^2$** (*"R-squared"*, or *"coefficient of determination"*). It is the most-quoted regression statistic in the world, and also one of the most misread. Let us build it from first principles, still on our `density ~ alcohol` worked example.

### Variance accounting

> **Callback to §1.5.4.** We met the **variance** of a single variable as the average squared deviation from its mean. The variance is a measure of *spread*.

For our outcome variable (`density` in this section), the **total spread** across the 1,599 wines is captured by the **total sum of squares**:

$$\mathrm{SS}_{\text{tot}} \;=\; \sum_{i=1}^{n} \bigl(y_i - \bar{y}\bigr)^2$$

This number — a sum, not yet divided by $n-1$ — is what we have to *"explain"*. Think of it as the total amount of variation in density we have to account for.

When we fit a regression, the model has its own opinion about each wine's density: $\hat{y}_i$, the prediction. Some of $\mathrm{SS}_{\text{tot}}$ ends up *explained* by the model (the predictions $\hat{y}_i$ vary across wines — and that variation accounts for some of the variation in $y$). The rest ends up *unexplained*, sitting in the residuals. That residual chunk has its own sum:

$$\mathrm{SS}_{\text{res}} \;=\; \sum_{i=1}^{n} \bigl(y_i - \hat{y}_i\bigr)^2$$

These are exactly the squared dashed-segment lengths from §4.10 — added up across the whole dataset (not just the 80-wine subsample).

### The definition

$$R^2 \;=\; 1 \;-\; \frac{\mathrm{SS}_{\text{res}}}{\mathrm{SS}_{\text{tot}}}$$

In words: *"one minus the fraction of the original spread that is still left over in the residuals"*. Equivalently: *"the fraction of the spread that the model **has** accounted for"*.

$R^2 = 1$ means the model accounts for *all* the spread — every data point sits exactly on the line, every residual is zero. $R^2 = 0$ means the model accounts for *none* of the spread — fitting the regression added nothing beyond just predicting $\bar{y}$ for every wine. Most real regressions live somewhere in between.

### Compute $R^2$ from scratch, then verify


In [ ]:
# All n=1599 wines, all predictions from the simple regression of density on alcohol.
y_hat_all = sm_model.predict(X_sm)

# Residuals (those dashed segments, computed for all 1,599 wines this time).
residuals_all = density - y_hat_all

# Total sum of squares: spread of the outcome (density) around its own mean.
density_mean = density.mean()
ss_tot = ((density - density_mean) ** 2).sum()

# Residual sum of squares: spread of the outcome around the regression line.
ss_res = (residuals_all ** 2).sum()

# R^2: one minus the fraction of spread that is still in the residuals.
r2_by_hand = 1 - ss_res / ss_tot

# Pearson r(alcohol, density) for the identity check below.
r_ad, _ = stats.pearsonr(alcohol, density)

# Print all the components plus the by-hand R^2.
print(f"SS_tot                       = {ss_tot:.6f}")
print(f"SS_res                       = {ss_res:.6f}")
print(f"R^2 by hand                  = {r2_by_hand:.6f}")
print()
print(f"R^2 from statsmodels         = {sm_model.rsquared:.6f}")
print(f"r^2 (square of r(alc, dens)) = {r_ad**2:.6f}")


You should see all three numbers agree at **$R^2 \approx 0.2462$**. Three things to notice:

1. The by-hand variance-decomposition calculation and `statsmodels`' built-in `rsquared` agree exactly.
2. **For a simple regression with one predictor, $R^2$ is literally $r^2$** — the square of the Pearson correlation. Our $r(\text{alcohol}, \text{density}) \approx -0.4962$ from §4.8, and $(-0.4962)^2 \approx 0.2462$. This is a tidy theorem: with one predictor, *"the variance explained by the line"* and *"the Pearson correlation squared"* are the same number. With multiple predictors (§4.14) that simple identity no longer holds; $R^2$ generalises but $r$ does not.
3. The interpretation: *"the simple regression of density on alcohol accounts for about 24.6% of the spread in wine densities. The remaining 75.4% lives in things alcohol alone does not capture"* — other physicochemical components (residual sugar, dissolved salts, etc.) all push and pull density too.

### What $R^2$ is *not*

- **$R^2$ is not a $p$-value.** It does not tell you whether the slope is statistically significant; it tells you how big the fit is. A regression with a tiny $R^2$ can still have a highly significant slope (with enough data); a regression with a large $R^2$ can still have an unreliable slope (with too little data).
- **$R^2$ is not a measure of causal power.** A model can have an $R^2$ of 0.9 because the predictor causes the outcome, **or** because both are driven by something else, **or** because the predictor is just a tautological restatement of the outcome. $R^2$ is silent about *why* the line fits.
- **$R^2$ is not a universal *"good enough"* threshold.** Whether 0.25 is *"good enough"* depends entirely on the domain. In a controlled physics experiment, 0.25 would be a scandalous failure of theory. In a behavioural panel study of wine tasting, 0.25 from a *single predictor* is a real and informative result. We will lift the multiple-regression $R^2$ on `quality` to about 0.34 with five predictors in §4.14.

> **Mini-recap of §4.11.** $R^2$ is one minus the residual sum of squares divided by the total sum of squares — the fraction of the outcome's spread the model accounts for. For one predictor, $R^2 = r^2$. **For density on alcohol alone, $R^2 \approx 0.25$ — alcohol explains roughly a quarter of the spread in wine density; three-quarters live elsewhere.** Now we have the full simple-regression toolkit (model, residuals, $R^2$, t-test on the slope). Time to turn it on the actual brief: what drives `quality`?


***
## 4.12 Partial correlation — built from residuals

> **Transition from the simple-regression worked example.** In §§4.9–4.11 we built the simple-regression machinery — fit, residuals, $R^2$, the t-test on the slope — on the continuous pair **`density ~ alcohol`**. That kept the residual visual in §4.10 clean and the dashed segments crisp, exactly the picture a beginner needs to lock the concept of *residual* into intuition. **Now we turn the machinery on the actual brief**: what drives `quality`? From this section onward, the residual scatters and the diagnostic plots will carry visible **horizontal banding** (when `quality` is on the y-axis of a scatter) or **diagonal stripes** (when residuals from a `quality` regression are plotted against predictions, as we will see in §4.14). That is the integer outcome reflecting through, *not* a model defect. **The concepts — residual, partial correlation, part correlation, multiple regression's *all-else-equal* interpretation — work identically; the pictures simply look stripier.**

Look back at the §4.8 correlation matrix. Two facts pop out:

- $r(\text{alcohol}, \text{quality}) \;\approx\; +0.476$ — what we have been working with.
- $r(\text{alcohol}, \text{density}) \;\approx\; -0.496$ — alcohol and density are **strongly** correlated. (Higher-alcohol wines are less dense, because ethanol is lighter than water.)
- $r(\text{density}, \text{quality}) \;\approx\; -0.175$ — density itself has a small negative correlation with quality.

This raises a natural and uncomfortable question:

> *"The alcohol/quality correlation might not be **about alcohol** at all. It might just be that high-alcohol wines tend to be low-density wines, and low-density wines happen to score better for some other reason. How much of the alcohol/quality story survives if we **strip out the density contribution** from both sides?"*

The answer is the **partial correlation** between alcohol and quality, **controlling for** density. There is a precise definition with formulas — but the most useful way to understand it is **conceptually**, using the residuals we built in §4.10.

### The recipe, in three steps

1. **Regress alcohol on density.** Compute residuals. *These residuals are the part of alcohol that density does not explain* — *"alcohol, with density's contribution removed"*.
2. **Regress quality on density.** Compute residuals. *These residuals are the part of quality that density does not explain*.
3. **Compute the Pearson correlation of those two residual columns.** *That* is the partial correlation between alcohol and quality, controlling for density.

That is the entire definition. No textbook formula needed.

### Why this works, in one paragraph

Partial correlation answers the question *"after removing the part of $X$ that comes along with $Z$, and the part of $Y$ that comes along with $Z$, how much linear relationship between $X$ and $Y$ is left?"* Each regression-and-residual operation **strips $Z$'s contribution out of one variable**. Correlating what is left says, in effect, *"both variables have been cleaned of any common $Z$ component — what they still share must be a relationship between $X$ and $Y$ that is genuinely independent of $Z$."*

### Compute it on the wine data


In [ ]:
# We will need the density column.
density = df["density"].values

# Step 1 — regress alcohol on density. Keep the residuals.
# Add the constant column for the intercept.
Xz_density = sm.add_constant(density)
model_alcohol_on_density = sm.OLS(alcohol, Xz_density).fit()
resid_alcohol = alcohol - model_alcohol_on_density.predict(Xz_density)

# Step 2 — regress quality on density. Keep the residuals.
model_quality_on_density = sm.OLS(quality, Xz_density).fit()
resid_quality = quality - model_quality_on_density.predict(Xz_density)

# Step 3 — Pearson correlation of the two residual columns.
r_partial, p_partial = stats.pearsonr(resid_alcohol, resid_quality)

# Print the raw and partial correlations side by side for comparison.
print(f"raw r(alcohol, quality)                   = {r_scipy:+.4f}")
print(f"partial r(alcohol, quality | density)     = {r_partial:+.4f}")
print(f"p-value for the partial correlation       = {p_partial:.3e}")


You should see the partial correlation **$r \approx +0.456$**, only slightly smaller than the raw correlation of **$+0.476$**. Interpretation: the alcohol/quality relationship is **almost entirely robust** to controlling for density. Most of it survives after removing density's contribution from both sides. That is reassuring evidence that the alcohol effect is not just a sneaky disguise of a density effect.

(If the partial had shrunk to near zero, the headline would have been very different: *"the alcohol/quality correlation is almost entirely **explained** by their common dependence on density."*)

### A picture worth a thousand words

Here is the **companion residual scatter** for this section. We plot the same residual visual from §4.10 — twice, side by side. On the left, density predicts alcohol; the dashed segments are the residuals from that fit. On the right, density predicts quality; same residual visual. Partial correlation is then nothing more than the **Pearson correlation between the two columns of dashed-segment lengths** (with sign).


In [ ]:
# Side-by-side residual visuals for the two intermediate regressions.
# We subsample 80 wines (same indices as §4.10) so the segments stay readable.
x_d = density[sample_idx]
y_alc = alcohol[sample_idx]
y_qual = quality[sample_idx]

# Predictions for those 80 wines from the two intermediate regressions.
yhat_alc  = model_alcohol_on_density.predict(sm.add_constant(x_d))
yhat_qual = model_quality_on_density.predict(sm.add_constant(x_d))

# Create the two-panel figure.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ---- Left panel: density -> alcohol ----
# Regression line across the full observed density range.
x_line = np.linspace(density.min(), density.max(), 200)
axes[0].plot(x_line, model_alcohol_on_density.predict(sm.add_constant(x_line)),
             color="red", linewidth=1.2)
# Dashed residual segments for the 80 sampled wines.
for xi, yi, yh in zip(x_d, y_alc, yhat_alc):
    axes[0].plot([xi, xi], [yi, yh], color="black", linestyle="--", linewidth=0.7)
# Actual data points (white-on-black) and predicted points (red).
axes[0].scatter(x_d, y_alc, s=60, facecolor="white", edgecolor="black", linewidth=1.0, zorder=3)
axes[0].scatter(x_d, yhat_alc, s=18, color="red", zorder=4)
axes[0].set_xlabel("density")
axes[0].set_ylabel("alcohol")
axes[0].set_title("Step 1: density → alcohol\n(residuals = part of alcohol that density does NOT explain)")

# ---- Right panel: density -> quality ----
axes[1].plot(x_line, model_quality_on_density.predict(sm.add_constant(x_line)),
             color="red", linewidth=1.2)
for xi, yi, yh in zip(x_d, y_qual, yhat_qual):
    axes[1].plot([xi, xi], [yi, yh], color="black", linestyle="--", linewidth=0.7)
axes[1].scatter(x_d, y_qual, s=60, facecolor="white", edgecolor="black", linewidth=1.0, zorder=3)
axes[1].scatter(x_d, yhat_qual, s=18, color="red", zorder=4)
axes[1].set_xlabel("density")
axes[1].set_ylabel("quality")
axes[1].set_title("Step 2: density → quality\n(residuals = part of quality that density does NOT explain)")

plt.tight_layout()
plt.show()


**What we see.** Two panels, same shape, both showing the by-now-familiar residual pattern. The **left** panel residuals — *"the part of alcohol that density does not explain"* — and the **right** panel residuals — *"the part of quality that density does not explain"* — are precisely the two columns of numbers we then correlated. Their Pearson correlation is the **+0.456** above. Visually, this number says *"after we strip away from both variables whatever density was carrying, the remaining alcohol-leftovers and the remaining quality-leftovers still move together strongly."*

> **Mini-recap of §4.12.** Partial correlation = correlation of residuals from regressing both variables on the control. For alcohol/quality controlling for density: **$r_{\text{partial}} \approx +0.456$**, only marginally smaller than the raw **+0.476**. The alcohol effect survives the density control. **Always express partial correlation as residuals first, then correlation second — that is what it literally is.**


***
## 4.13 Part (semi-partial) correlation — the *"all else equal"* primitive

Partial correlation strips the control variable's contribution out of **both** sides — predictor and outcome. Sometimes that is what you want, and sometimes it is not.

A second, slightly different question is:

> *"Holding density fixed in the predictor only, how much of `quality`'s variation does the **remaining**, density-free part of `alcohol` capture?"*

The answer is the **part correlation**, also called the **semi-partial correlation**. The recipe is the partial-correlation recipe, with one step skipped: **residualise only the predictor, not the outcome**.

### The two-line recipe

1. **Regress the predictor (alcohol) on the control (density). Keep the residuals** — *"the part of alcohol that density does not explain"*. (This is **Step 1** from §4.12 — we have it already.)
2. **Correlate that residual column with the original outcome (quality), unmodified.** That correlation is the part / semi-partial correlation.

Notice the asymmetry: in partial correlation we residualise both; in part correlation we residualise only one. The two numbers will usually be slightly different.

### Compute it


In [ ]:
# We already have resid_alcohol from §4.12 — "alcohol stripped of density".
# Just correlate it with the original, unmodified quality.
r_part, p_part = stats.pearsonr(quality, resid_alcohol)

# Show all three flavours side by side.
print(f"raw correlation         r(alcohol, quality)              = {r_scipy:+.4f}")
print(f"partial (§4.12)         r(alcohol, quality | density)    = {r_partial:+.4f}")
print(f"part / semi-partial     r(quality, alcohol minus density)= {r_part:+.4f}")
print(f"  (predictor residualised, outcome untouched)")


You should see **part $r \approx +0.448$** — slightly smaller than both the raw and the partial. The three numbers are close to each other in this particular case because alcohol and density together explain only a moderate share of quality (and density alone explains very little); but in datasets with stronger confounding the three values can differ substantially.

### Why this matters — *"all else equal"*, made precise

Read the part correlation aloud: *"the correlation between `quality` (as is) and the part of `alcohol` that has nothing to do with `density`."* That phrase — *"the part of $x$ that has nothing to do with the other predictors"* — is **exactly what the slope coefficient in a multiple regression captures**.

> A multiple regression coefficient $\hat{\beta}_k$ is not *"the relationship of $x_k$ with $y$"*. It is *"the relationship of the part of $x_k$ that is **independent of all the other predictors in the model** with $y$"*. That is what the textbook phrase **"all else equal"** literally means: *"hold the other predictors fixed by residualising them out"*. The part correlation in §4.13 is the same idea applied with one control; multiple regression in §4.14 generalises it to many controls at once.

This is the conceptual unlock for the next section. Without the residual visual from §4.10 and the part-correlation construction here, multiple regression's *"all else equal"* claim would just be a slogan to memorise. With them, it is something you can *picture*: a coefficient in a multiple regression is the slope of $y$ on the **residualised** version of the corresponding predictor.

> **Mini-recap of §4.13.** Part correlation = correlation of the outcome (as is) with the **residualised** predictor (controls stripped out). It is what makes multiple regression's *"all else equal"* coefficient interpretation precise.


***
## 4.14 Multiple linear regression — many drivers at once

Now we are ready for the question we have been circling: **fit a model with several predictors at once**, and read the coefficients in the *"all else equal"* language of §4.13. This is the workhorse model behind the drivers memo we will hand the winemaker.

### The model

For $p$ predictors:

$$y_i \;=\; \beta_0 \;+\; \beta_1 \, x_{1i} \;+\; \beta_2 \, x_{2i} \;+\; \dots \;+\; \beta_p \, x_{pi} \;+\; \varepsilon_i$$

Reading every symbol:

- $y_i$ — the $i$-th outcome (the wine's `quality`).
- $x_{ki}$ — the $i$-th observation of the $k$-th predictor.
- $\beta_0$ — intercept, as before.
- $\beta_k$ — the slope on the $k$-th predictor. **In multiple regression**, $\beta_k$ is interpreted as *"the change in $y$ associated with a one-unit increase in $x_k$, **holding all the other predictors fixed**"*. The *"all else equal"* clause is no longer optional — it is built into what the coefficient literally represents (§4.13).
- $\varepsilon_i$ — error term.
- $p$ — the number of predictors. (Not to be confused with the $p$-value!)

### Which predictors will we include?

We pick the five candidates with the largest absolute correlations with `quality` from the §4.8 ranking, skipping highly redundant ones for now: **alcohol**, **volatile acidity**, **sulphates**, **citric acid**, and **pH**. (We deliberately keep the predictor list small so the table is readable — in a real drivers analysis you might iterate on the predictor set; today the focus is the machinery.)


In [ ]:
# Pick the five predictors for our multiple regression.
predictor_names = ["alcohol", "volatile acidity", "sulphates", "citric acid", "pH"]
X_multi = df[predictor_names].values

# Add the constant column for the intercept; this is the design matrix.
X_multi_const = sm.add_constant(X_multi)

# Fit the OLS multiple regression of quality on these five predictors.
multi_model = sm.OLS(quality, X_multi_const).fit()

# Print the full summary.
print(multi_model.summary())


### Reading the summary table

Five things to lock in from the output.

**1. The coefficients on the original scale.**

| predictor | $\hat{\beta}$ | interpretation (*all else equal*) |
|---|---:|---|
| `alcohol` | $\approx +0.327$ | each +1% alcohol associated with $+0.33$ quality points, **holding the other four fixed**. |
| `volatile acidity` | $\approx -1.284$ | each +1 unit volatile acidity associated with $-1.28$ quality points. **The strongest single effect on the original scale.** |
| `sulphates` | $\approx +0.673$ | each +1 unit sulphates associated with $+0.67$ quality points. |
| `citric acid` | $\approx -0.297$ | each +1 unit citric acid associated with $-0.30$ quality points (a sign flip from its **positive** raw correlation in §4.8 — see *"sign flips"* below). |
| `pH` | $\approx -0.475$ | each +1 unit pH associated with $-0.47$ quality points. |

**2. The $t$- and $p$-values.** Each coefficient has its own $t$-statistic — the coefficient divided by its own standard error — and a two-sided $p$-value. The same machinery as a one-sample t-test, applied to each row. All five predictors here have very small $p$-values (the largest is `citric acid` at about 0.014); we **reject the null** that each individual coefficient is zero. *"Statistical significance"* on a coefficient, however, says *nothing* about practical importance — it just says the effect is unlikely to be sampling noise.

**3. The model-wide $R^2$.** About **0.341**. The five-predictor model accounts for roughly **34.1% of the variance** in panel quality scores — a meaningful step up from the 22.7% achieved by `alcohol` alone in §4.9. Two-thirds of the spread still live in things our five predictors do not capture; this is realistic for any panel-rated subjective outcome.

**4. The model-wide $F$-statistic.** Tests the joint null *"all slopes are zero simultaneously"*. With $F \approx 165$ and $p \approx 10^{-141}$ we reject that joint null with enormous margin — there is *some* combination of these five that genuinely associates with `quality`.

**5. Sign flips between the raw correlation and the multiple-regression coefficient.** Look at `citric acid`: its raw correlation with quality (§4.8) was $+0.226$, but its multiple-regression coefficient is $-0.297$. The sign reversed. This is **not a mistake** — it is one of the most important phenomena in multiple regression and it deserves a paragraph of its own.

### Why coefficients can flip sign — *"all else equal" in action*

The raw correlation of `citric acid` with `quality` is **positive** — wines with more citric acid tend to score higher. But citric acid is also strongly correlated with **`fixed acidity`** and **`pH`** (chemistry: citric acid pushes pH down). Once we put `pH` *into* the regression alongside `citric acid`, the coefficient on `citric acid` is no longer *"the relationship between citric acid and quality"* — it is *"the relationship between the part of `citric acid` that is **not** carried by `pH` (and the other predictors) and quality"*. That residualised version of citric acid moves *differently* with quality than the raw variable does, and so the sign of the coefficient can flip. The §4.13 part-correlation machinery is exactly the conceptual hook for this. The raw correlation and the multiple-regression coefficient answer **different questions**; they should not be expected to agree, and when they do not it usually means *"there is a confound, and the multiple regression is correcting for it."*

This is also why the *"all else equal"* clause is not a polite hedge — **it is the only valid interpretation**. Say it out loud, every time, when reading a coefficient from a multiple regression.

### What each row's t-test is actually comparing — *"all else equal" again*

> **Callback to §4.9's t-test sub-section.** Every row of the coefficient table is its own t-test. The machinery is identical to §4.9: each $\hat{\beta}_k$ is compared to a null value of zero, in units of its own standard error, and the p-value is the two-tailed area beyond $|t|$ in a t-distribution. But the *meaning of the null* changes once there is more than one predictor in the model.

In the simple regression of §4.9, the null was *"alcohol has no linear association with density"*. In the multiple regression here, the null for the `alcohol` row is **not** *"alcohol has no association with quality"* — it is:

> *"the **all-else-equal** slope of `quality` on `alcohol` is zero"* — that is, *"once volatile acidity, sulphates, citric acid, and pH are already in the model, the part of alcohol that is independent of those four predictors contributes nothing to quality."*

This is why the t-test on a multiple-regression coefficient is a **fundamentally different question** from the t-test on a simple-regression coefficient — and from a test on the raw correlation. It is the **§4.13 part-correlation question, phrased as a hypothesis test**.

The **degrees of freedom** for each row's t-distribution are $n - p - 1 = 1{,}593$ (one observation per data point, minus $p + 1 = 6$ parameters — five slopes plus the intercept). For this sample size the t-distribution is essentially Normal, so we can read the t-statistics directly as *"standard errors away from zero"*.

For our wine model, every row rejects its own null:

- `alcohol`: $t \approx +19.76$, $p \approx 10^{-78}$. Reject with huge margin. The *all-else-equal* effect of alcohol on quality is overwhelmingly different from zero.
- `volatile acidity`: $t \approx -11.42$, $p \approx 10^{-29}$. Reject.
- `sulphates`: $t \approx +6.54$, $p \approx 10^{-11}$. Reject.
- `pH`: $t \approx -3.55$, $p \approx 0.0004$. Reject.
- `citric acid`: $t \approx -2.47$, $p \approx 0.014$. Reject at $\alpha = 0.05$, but just barely — **and the sign is the opposite of citric acid's raw correlation with quality** (raw $r$ was $+0.226$ in §4.8; here $\hat{\beta} \approx -0.30$). We unpack that sign flip in the next paragraph.


### Standardised coefficients — comparing across drivers

The coefficients above are on each predictor's **original scale**. That makes them readable in the predictor's native units (*"per +1% alcohol"*, *"per +1 unit of pH"*), but it makes them *uncomparable across predictors*: a coefficient of $-1.28$ on volatile acidity is **not** automatically a bigger effect than $+0.33$ on alcohol — the two predictors are measured in completely different units.

To rank drivers fairly, we re-fit the regression after **$z$-scoring every variable** (predictors and outcome). The coefficients we get back are the **standardised coefficients** — each one says *"how many standard deviations of $y$ does the model associate with a one-standard-deviation increase in $x_k$, all else equal"*. They are directly comparable across predictors.


In [ ]:
# z-score every predictor (subtract its mean, divide by its sample SD) and the outcome.
X_z = (df[predictor_names] - df[predictor_names].mean()) / df[predictor_names].std(ddof=1)
y_z = (quality - quality_mean) / quality_sd

# Fit the regression on the z-scored data, no intercept needed (it would be exactly zero by construction).
std_model = sm.OLS(y_z, X_z.values).fit()

# Report the standardised coefficients.
print("Standardised regression coefficients (z-scored predictors and outcome):")
print(f"{'predictor':22s} {'beta_std':>10s}")
for name, b in zip(predictor_names, std_model.params):
    print(f"  {name:20s} {b:>+10.4f}")


### Reading the standardised ranking

In standardised units, the five predictors rank cleanly by absolute effect size:

1. **`alcohol`**: standardised $\beta \approx +0.43$. *"A +1 SD increase in alcohol is associated with a +0.43 SD increase in quality, all else equal."* The strongest single driver.
2. **`volatile acidity`**: standardised $\beta \approx -0.28$. Second-strongest, in the expected (negative) direction.
3. **`sulphates`**: standardised $\beta \approx +0.14$. A real but smaller positive contribution.
4. **`pH`**: standardised $\beta \approx -0.09$.
5. **`citric acid`**: standardised $\beta \approx -0.07$. Smallest absolute effect, despite being statistically significant.

That ranking — **alcohol, volatile acidity, sulphates** as the meaningful trio, with `pH` and `citric acid` as minor contributors — is the **first sentence of the drivers memo we will hand the winemaker** in §4.17.


### Multicollinearity check — *VIF*

There is one diagnostic we should run before trusting any of these multiple-regression coefficients: **VIF (variance inflation factor)**. The concern: when two predictors are strongly correlated with **each other**, the regression cannot cleanly attribute the outcome's movement to one versus the other, and the coefficients become unstable — small data changes can push them around wildly, and their standard errors balloon.

VIF measures, for each predictor $x_k$, *how much the standard error of $\hat{\beta}_k$ is inflated, relative to a hypothetical world in which $x_k$ were uncorrelated with all the other predictors in the model*. The computation: regress $x_k$ on the other predictors, take the resulting $R^2$ of that *predictor-on-predictors* regression, and:

$$\mathrm{VIF}_k \;=\; \frac{1}{1 - R^2_k}$$

A VIF of 1.0 means *"no inflation — this predictor is uncorrelated with the others"*. A VIF of 2.0 means *"the standard error is $\sqrt{2} \approx 1.4 \times$ wider than it would otherwise be"*. The most-cited rules of thumb: a VIF above 5 is *"worth a careful look"*; above 10 is *"likely a real problem; consider dropping or combining predictors."*


In [ ]:
# Compute VIF for each of our five predictors.
# variance_inflation_factor wants the FULL design matrix (with the constant) and a column index.
# Indices 1..5 correspond to our five predictors (index 0 is the constant column).
print(f"{'predictor':22s} {'VIF':>8s}")
for i, name in enumerate(predictor_names, start=1):
    vif_k = variance_inflation_factor(X_multi_const, i)
    print(f"  {name:20s} {vif_k:>8.3f}")


All five VIFs are between 1.1 and 2.1 — **comfortably below any rule-of-thumb concern threshold**. The predictors carry some overlapping information (especially `citric acid` with `pH`, as we already saw chemically), but not enough to make any of the coefficient estimates unstable. The multiple-regression interpretation in the previous section is safe to report.

### Residual diagnostics

> **Callback to §4.10.** The residual is the unit of currency. Now that the model is fit, we look at the residuals **as a column** to check the assumptions the OLS machinery relies on.

Two quick diagnostic pictures:

1. **Predicted vs residual** — should look like a structureless cloud around zero. Any **fan shape** suggests the residual variance grows with the prediction (heteroscedasticity); any **curve** suggests the linear form is wrong.
2. **Histogram of residuals** — should look roughly bell-shaped, centred on zero. The OLS standard errors and $p$-values are slightly more trustworthy when this holds, though for $n = 1599$ the central limit theorem makes the inference robust to mild departures.


In [ ]:
# Predictions and residuals for all 1,599 wines from the multiple model.
multi_predictions = multi_model.predict(X_multi_const)
multi_residuals = quality - multi_predictions

# Two-panel diagnostic figure.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left panel: predicted vs residual scatter.
axes[0].scatter(multi_predictions, multi_residuals, s=15, alpha=0.4, color="black")
axes[0].axhline(0, color="red", linewidth=1)
axes[0].set_xlabel("predicted quality (from the multiple regression)")
axes[0].set_ylabel("residual = actual − predicted")
axes[0].set_title("Predicted vs residual (should be structureless)")

# Right panel: histogram of residuals.
axes[1].hist(multi_residuals, bins=40, edgecolor="black")
axes[1].axvline(0, color="red", linewidth=1)
axes[1].set_xlabel("residual = actual − predicted")
axes[1].set_ylabel("number of wines")
axes[1].set_title("Histogram of residuals (should be roughly bell-shaped, centred on 0)")

plt.tight_layout()
plt.show()


**What we see — and what to make of the structure that does appear.**

**The predicted-vs-residual plot looks far from "structureless"** — it shows six clearly visible **diagonal stripes**. This is not a model defect. It is a direct geometric consequence of `quality` being an integer in 3..8:

- For every wine with actual `quality = k` (where $k \in \{3, 4, 5, 6, 7, 8\}$), the residual is $\text{residual} = k - \hat{y}$.
- That is a straight line in `(predicted, residual)` space — slope $-1$, intercept $k$.
- Six possible values of $k$ give **six parallel diagonal lines, each of slope $-1$**.

> **§4.12 forewarned this.** Whenever `quality` is the outcome of a regression, residual plots will carry this kind of integer-banding structure — **the outcome reflecting through, not a sign that the linear model is wrong.** A continuous outcome (such as `density` in our §4.9–§4.11 worked example) would produce the structureless cloud the textbook diagnostic asks for; an integer outcome cannot, *by construction*.

What we are actually looking for in this diagnostic — once we mentally remove the diagonal-grid artifact — is three things, all of which hold here:

1. **Do the stripes stay roughly centred on zero across the predicted-quality range?** Yes — the red zero-line passes through the middle of the cloud.
2. **Do the stripes curve, suggesting a missing non-linear term?** No — they remain straight.
3. **Does the vertical spread of points within a band fan out as predictions grow, suggesting heteroscedasticity?** No — the spread looks roughly even.

**The histogram of residuals** is the cleaner picture: roughly bell-shaped, centred on zero, with slightly heavier-than-Normal tails. Acceptable for OLS inference at this sample size.

The diagnostic picture is clean enough — once interpreted through the integer-outcome lens — that we can take the standardised coefficient ranking and the $R^2$ at face value when writing the memo.


***
## 4.15 Dummy coding — adding a categorical predictor

The multiple regression in §4.14 had five **continuous** predictors. But the world is full of **categorical** drivers — region, supplier, season, A/B variant, **wine variety** — and the regression machinery does not natively understand text labels. **Dummy coding** is the technique that fixes that, and it is the technique you will reach for almost weekly in real analyst work.

> **Callback to the §4.17 memo's *Open questions* list.** The stakeholder memo at the end of this session asks *"does the same driver ranking hold on the white-wine archive?"* This section makes a first pass at that question — by **combining red and white wines into a single dataset** and asking, with a regression that accepts a categorical predictor, *"is the wine type (red vs. white) itself a driver of panel quality, all else equal?"*

The data is already in your `_data/wine_quality/` folder. Alongside `winequality-red.csv` sits its sibling `winequality-white.csv` — **4,898 white wines**, identical column structure to the red dataset, same eleven physicochemical measurements, same 0–10 panel `quality` score. Stacked together with a new `wine_type` column tacked on, we get **6,497 wines of two clearly different wine styles** — and a genuine binary categorical predictor that exists in the data **as-given**, no manufactured binning required.


In [ ]:
# Load the white-wine CSV using the same `;`-separator trick we used for the reds.
white_wines = pd.read_csv(
    "../_data/wine_quality/winequality-white.csv",
    sep=";"
)

# Tag every row with its type so we can tell them apart after concatenation.
# We make a copy of the red-wine DataFrame `df` (loaded back in §4.3) so we do not mutate the original.
red_wines = df.copy()
red_wines["wine_type"]   = "red"
white_wines["wine_type"] = "white"

# Stack reds on top of whites into one combined DataFrame.
# `ignore_index=True` re-numbers the rows 0..N-1 (otherwise the original red indices would clash with white).
combined = pd.concat([red_wines, white_wines], ignore_index=True)

# Confirm the row counts so we know the concatenation worked.
print(f"reds:     {(combined['wine_type'] == 'red').sum():,}")
print(f"whites:   {(combined['wine_type'] == 'white').sum():,}")
print(f"combined: {len(combined):,}")


### A look at the raw per-type means, before any regression

Before we touch dummy coding, a quick look at the **raw, unconditional** per-type means. These tell us what the comparison would look like *without any regression at all* — just the average panel score and the average alcohol per type. **Keep these numbers in mind**: the regression coefficient on the dummy below will tell us what survives **after controlling for alcohol**, which is generally not the same as the raw gap.


In [ ]:
# Group the combined DataFrame by wine_type and compute three summary numbers per group:
# the count (how many wines of each type), the mean panel quality, and the mean alcohol.
per_type_summary = combined.groupby("wine_type").agg(
    n            = ("quality", "size"),
    mean_quality = ("quality", "mean"),
    mean_alcohol = ("alcohol", "mean"),
).round(3)

# Show the summary table.
per_type_summary


You should see that **white wines score on average about +0.24 panel points higher than red wines** (5.878 vs 5.636), and their mean alcohol levels are very similar (10.51% vs 10.42%). Whether that raw +0.24 difference *survives controlling for alcohol* — and so represents a genuine wine-type effect — is exactly the question dummy coding lets us answer.

### The encoding problem

A regression coefficient is a number — you cannot multiply *"red"* or *"white"* by anything. To put `wine_type` into a regression, we have to convert it to numbers. The obvious naïve approach — *"give each category a number, like red = 0, white = 1, rosé = 2"* — actually works for our binary case (because there is only one gap to encode), but it **fails badly** for three or more categories, because it forces the regression to treat the categories as if they were on an ordered numerical scale with equal spacing. *"Rosé is twice as wine-type-y as white"* is not a sentence anyone wants to say.

### Dummy coding — the solution

The fix is **dummy coding**, also called *one-hot encoding* in the machine-learning world. For a categorical variable with $k$ levels, we create **$k - 1$** binary indicator columns. Each indicator column is **1** for the rows that belong to its level and **0** otherwise. The category that *does not* get its own column is the **baseline**: when all the dummies in a row are 0, the row implicitly belongs to the baseline.

For our binary case ($k = 2$, levels $\{$red, white$\}$), we need **just one dummy column**. Let us pick **red as the baseline** and create a single column `is_white`:

- `is_white = 1` for white wines.
- `is_white = 0` for red wines (the baseline — no column of its own).

### Why $k - 1$ instead of $k$? — The dummy variable trap

If we had created **both** columns (`is_red` AND `is_white`), they would sum to **1 for every row** (every wine is either red or white). A column of 1s is **already in the design matrix as the intercept column**. The two dummies plus the intercept would be perfectly redundant — there would be infinitely many equally-good coefficient choices, and the regression would be unable to estimate any of them. This redundancy is called the **dummy variable trap**, and dropping one dummy (equivalently: choosing a baseline) avoids it. `pandas` and `scikit-learn` both implement this correctly when you use their helper functions (`pd.get_dummies(..., drop_first=True)`, or `C(variable)` in a statsmodels formula).


In [ ]:
# Create the dummy column.
# (combined["wine_type"] == "white") returns a Boolean Series (True/False);
# `.astype(int)` converts True to 1 and False to 0 — exactly the binary indicator we want.
combined["is_white"] = (combined["wine_type"] == "white").astype(int)

# Sanity-check: print the first 3 rows of each wine type to confirm the encoding is correct.
print("Red wines (is_white should be 0):")
print(combined[combined["wine_type"] == "red"][["wine_type", "is_white"]].head(3))
print()
print("White wines (is_white should be 1):")
print(combined[combined["wine_type"] == "white"][["wine_type", "is_white"]].head(3))


### Fit the model

We will fit a multiple regression with **one continuous predictor (`alcohol`) and one dummy predictor (`is_white`)**:

$$\text{quality}_i \;=\; \beta_0 \;+\; \beta_{\text{alc}} \cdot \text{alcohol}_i \;+\; \beta_{\text{white}} \cdot \text{is\_white}_i \;+\; \varepsilon_i$$

Reading every symbol:

- $\text{quality}_i$, $\text{alcohol}_i$ — the $i$-th wine's panel quality and alcohol, as before.
- $\text{is\_white}_i$ — the dummy column we just built; **1 if the $i$-th wine is white, 0 if red**.
- $\beta_0$ — intercept, as before.
- $\beta_{\text{alc}}$ — the alcohol slope. **The same coefficient for both wine types** — that is the *parallel-slopes* assumption that additive dummy coding makes. (Allowing different slopes per type would require an **interaction term**, which we deliberately leave for a later session — for an introductory course, the parallel-slopes additive case is the foundational one to master first.)
- $\beta_{\text{white}}$ — the dummy coefficient. **This is the new thing to interpret carefully** — it will tell us the *all-else-equal* difference in predicted quality between a white wine and a red wine.

We fit it with the same `sm.OLS` machinery we used for the simple regression in §4.9 and the multiple regression in §4.14.


In [ ]:
# Build the design matrix: a column of 1s for the intercept, then alcohol, then is_white.
# sm.add_constant prepends the constant column for the intercept.
X_dummy = sm.add_constant(combined[["alcohol", "is_white"]].values)

# Fit the OLS regression: outcome is quality, predictors are alcohol + is_white.
dummy_model = sm.OLS(combined["quality"].values, X_dummy).fit()

# Print the full summary so we can read every coefficient with its SE, t, and p.
print(dummy_model.summary())


### Reading the three coefficients

Three coefficients, three interpretations.

**1. Intercept $\hat{\beta}_0 \approx +2.272$.** Mathematically: the predicted quality for a wine with `alcohol = 0` AND `is_white = 0` — i.e., a **red** wine extrapolated to zero alcohol. As in §4.9, this is not literally meaningful (no wine has 0% alcohol). Its job is to anchor the predicted line so the other coefficients carry their intended interpretation.

**2. Alcohol slope $\hat{\beta}_{\text{alc}} \approx +0.323$.** Per +1% alcohol, predicted quality rises by about 0.32 panel points — **for either wine type**. This is the parallel-slopes interpretation: the dummy-coded model assumes alcohol affects quality at the same rate for reds and for whites. (Whether that assumption is actually true is a separate question, one we are deferring to a later session.)

**3. Dummy coefficient $\hat{\beta}_{\text{white}} \approx +0.212$ — the new thing.** This is the **all-else-equal difference between a white wine and a red wine, holding alcohol fixed**. In plain English: *"two wines with identical alcohol content — one red, one white — the white one is predicted to score about 0.21 panel points higher."*

The t-test on $\hat{\beta}_{\text{white}}$ (same machinery as §4.9's t-test sub-section, applied per coefficient as in §4.14's per-row callback): $t \approx 9.49$, $p \approx 3 \times 10^{-21}$. We **reject the null** *"once alcohol is in the model, wine type adds nothing"* — the wine-type difference is far too large to plausibly be sampling noise.

#### Numerical sanity check — the dummy coefficient *is* the vertical gap between two parallel lines

Let us verify the *"all else equal"* interpretation numerically. Pick three alcohol levels and compute the predicted quality for a red and a white wine at each, by hand from the three coefficients:

| alcohol | pred. red ($\hat{\beta}_0 + \hat{\beta}_{\text{alc}} \cdot a + \hat{\beta}_{\text{white}} \cdot 0$) | pred. white ($\hat{\beta}_0 + \hat{\beta}_{\text{alc}} \cdot a + \hat{\beta}_{\text{white}} \cdot 1$) | difference (white − red) |
|---:|---:|---:|---:|
| 9.0% | $2.272 + 0.323 \cdot 9.0 + 0.212 \cdot 0 \;\approx\; 5.177$ | $2.272 + 0.323 \cdot 9.0 + 0.212 \cdot 1 \;\approx\; 5.389$ | **+0.2124** |
| 11.0% | $\;\approx\; 5.822$ | $\;\approx\; 6.035$ | **+0.2124** |
| 13.0% | $\;\approx\; 6.468$ | $\;\approx\; 6.680$ | **+0.2124** |

The difference is **exactly the same at every alcohol level** — equal to the dummy coefficient $\hat{\beta}_{\text{white}} \approx +0.2124$. That is what *"all else equal"* literally means for a dummy: **the dummy coefficient is the vertical gap between two parallel lines** — one for each value of the dummy. The next picture makes this geometric.


In [ ]:
# Build the picture: scatter coloured by wine type, plus the two parallel regression lines from our fit.

# Use a default_rng for the y-jitter (same display trick as §4.5, since `quality` is integer 3..8).
rng = np.random.default_rng(42)

# Split the data into red and white subsamples so we can colour them differently.
red_mask   = combined["wine_type"] == "red"
white_mask = combined["wine_type"] == "white"
red_alc    = combined.loc[red_mask,   "alcohol"].values
white_alc  = combined.loc[white_mask, "alcohol"].values

# Apply small vertical jitter to the integer quality values, FOR DISPLAY ONLY (the fit was on the un-jittered data).
red_q   = combined.loc[red_mask,   "quality"].values + rng.uniform(-0.25, 0.25, size=red_mask.sum())
white_q = combined.loc[white_mask, "quality"].values + rng.uniform(-0.25, 0.25, size=white_mask.sum())

# Set up the figure.
plt.figure(figsize=(11, 6))

# Scatter: reds in crimson, whites in steelblue, both with low alpha so density shows through.
plt.scatter(red_alc,   red_q,   color="crimson",   alpha=0.25, s=12, label="red wines (n = 1,599)")
plt.scatter(white_alc, white_q, color="steelblue", alpha=0.15, s=12, label="white wines (n = 4,898)")

# Two parallel regression lines from the fitted model, drawn across the observed alcohol range.
# y_line_red is the prediction with is_white = 0; y_line_white is the prediction with is_white = 1.
x_line       = np.linspace(combined["alcohol"].min(), combined["alcohol"].max(), 200)
y_line_red   = dummy_model.params[0] + dummy_model.params[1] * x_line + dummy_model.params[2] * 0
y_line_white = dummy_model.params[0] + dummy_model.params[1] * x_line + dummy_model.params[2] * 1

# Plot the two lines on top of the scatter.
plt.plot(x_line, y_line_red,   color="crimson",   linewidth=2.0, label="model: red wines (is_white = 0)")
plt.plot(x_line, y_line_white, color="steelblue", linewidth=2.0, label="model: white wines (is_white = 1)")

# Labels and title.
plt.xlabel("alcohol (% by volume)")
plt.ylabel("quality (panel score, 0–10; jittered for display)")
plt.title("Dummy-coded regression: parallel slopes, separated by the dummy coefficient (+0.212)")
plt.legend(loc="upper left")
plt.tight_layout()
plt.show()


### What we see

Two parallel lines, one per wine type. They share the **same slope** (the +0.323 alcohol coefficient) and are separated by a **constant vertical gap** equal to the dummy coefficient (+0.212). The crimson cloud (reds) sits, on average, slightly below the steelblue cloud (whites), and the steelblue line lies above the crimson line by *exactly the same amount everywhere* — *"all else equal, white wines are predicted about 0.21 panel points higher than reds"*.

This is what an additive dummy in a linear model looks like geometrically: **two parallel lines whose vertical separation is the dummy coefficient**. Add a third wine type (a second dummy), and you would get **three** parallel lines, each separated from the baseline by its own dummy coefficient.

### Raw mean gap vs all-else-equal gap

> **Callback to §4.13.** The dummy coefficient is the **§4.13 part-correlation idea applied to a categorical predictor**: it answers *"after controlling for alcohol, how much of the quality difference between whites and reds is left?"*

The raw per-type comparison earlier said: white wines score 5.878 on average, reds 5.636 — a raw mean difference of **+0.242** panel points. The dummy regression says the all-else-equal difference is **+0.212** panel points. The two are close because reds and whites have similar mean alcohol (10.42% vs 10.51%) — there is not much for the regression to *"correct away."* In a domain where the categorical predictor was strongly **confounded** with a continuous one — e.g., if white wines were systematically much higher in alcohol than reds — the raw and the all-else-equal numbers could differ substantially, and **the regression's number would be the honest one to report** in the memo.


In [ ]:
# Re-fit the same model, but with WHITE as the baseline (we drop is_white and use is_red instead).
# This is purely a demonstration that the model fit does not depend on which level is the baseline.
combined["is_red"] = (combined["wine_type"] == "red").astype(int)
X_alt = sm.add_constant(combined[["alcohol", "is_red"]].values)
alt_model = sm.OLS(combined["quality"].values, X_alt).fit()

# Print the three coefficients side by side with the original fit so the comparison is clean.
print(f"{'coefficient':22s}  {'red as baseline':>18s}  {'white as baseline':>20s}")
print(f"{'intercept':22s}  {dummy_model.params[0]:>18.4f}  {alt_model.params[0]:>20.4f}")
print(f"{'alcohol slope':22s}  {dummy_model.params[1]:>+18.4f}  {alt_model.params[1]:>+20.4f}")
print(f"{'dummy coefficient':22s}  {dummy_model.params[2]:>+18.4f}  {alt_model.params[2]:>+20.4f}")
print()
print(f"R^2 (red baseline):   {dummy_model.rsquared:.4f}")
print(f"R^2 (white baseline): {alt_model.rsquared:.4f}   (identical)")


### Choice of baseline — interpretation flips, fit is unchanged

The two coefficient tables tell the same story in mirror image:

- The **alcohol slope** is identical in both models: +0.3228 either way.
- The **dummy coefficient** has the **same magnitude but opposite sign**: +0.2124 (white-vs-red) becomes −0.2124 (red-vs-white). This is *exactly the same statement* — *"whites score 0.21 above reds"* and *"reds score 0.21 below whites"* are the same fact.
- The **intercept** absorbs the shift: it changes from +2.272 to +2.484 to compensate (the +0.2124 gap has to live somewhere when we switch the baseline).
- The model's **predictions are wine-by-wine identical**, and the **$R^2$ is identical**. The model fit does not depend on which level you pick as the baseline — only the *interpretation* of the dummy coefficient changes.

**Practical advice for the analyst.** Pick the baseline that makes the interpretation easiest for your stakeholder. In our case, *"white wines score about 0.21 higher than reds, all else equal"* is the more natural sentence than its mirror, so leaving **red as the baseline** (the `is_white` dummy) is the better choice for the memo.

### Multi-category extension — one paragraph

For a categorical predictor with **more than two** levels — say, a `region` variable with levels $\{$Bordeaux, Burgundy, Loire, Rhône$\}$ — we create $k - 1 = 3$ dummies. One region (say, Bordeaux) is the baseline; the regression returns three coefficients (`is_Burgundy`, `is_Loire`, `is_Rhône`), each one interpreted as *"the all-else-equal difference vs. the Bordeaux baseline."* In `pandas`, `pd.get_dummies(combined["region"], drop_first=True)` constructs these columns for you; in `statsmodels`, the formula `quality ~ alcohol + C(region)` does the same thing implicitly. The intuition is identical to the binary case — each dummy is the **vertical gap** between its level's regression line and the baseline's regression line, all lines sharing the same continuous-predictor slope.

> **Mini-recap of §4.15.** A categorical predictor enters a regression via **dummy coding**: $k - 1$ binary indicator columns, with one category as the baseline. The dummy coefficient is the **all-else-equal difference in predicted outcome between that category and the baseline** — geometrically, the vertical gap between two parallel regression lines. The choice of baseline changes the interpretation, never the fit. For wine quality on alcohol + wine-type: $\hat{\beta}_{\text{white}} \approx +0.212$ (red baseline), $t \approx 9.49$, $p \approx 3 \times 10^{-21}$ — white wines score about 0.21 panel points higher than reds, all else equal. A small but highly statistically significant effect. With this tool added to your kit, the memo's *"does the same driver ranking hold on the white-wine archive?"* open question gets its first principled answer.


***
## 4.16 Our API calls — Anthropic tool use for the drivers memo

> **Callback to §1.11, §2.13, §3.12.** Session 01 made plain-text Anthropic calls. Session 02 asked Anthropic for a JSON outline and validated structure with Python-side checks. Session 03 stepped up to **schema-enforced** structured output, using **Anthropic's tool use** feature with a Pydantic schema (§3.12) — the model literally could not return something that did not match the schema. **Same pattern this week**, applied to the drivers memo.

> **Why Anthropic only.** This course uses **Anthropic Claude exclusively** — there is no second provider, no second SDK, no second API key. Anthropic's Messages API plus tool use gives us everything we need: plain-text generation for language work, and schema-enforced output for structured deliverables. (This is course **Principle 20** in the instructor's notes — and the reason §3.12's earlier OpenAI references were rewritten.)

We will make **two API calls** this week, both to Anthropic:

1. **Call 1 — schema-enforced drivers plan.** Anthropic generates the structured plan the winemaker asked for, as a JSON object that *conforms to a Pydantic schema we define in Python*. The schema has three required fields: `key_findings`, `caveats`, and `data_that_would_strengthen_conclusion`.
2. **Call 2 — plain-text headline paragraph.** Anthropic drafts the *"Headline findings"* paragraph of the stakeholder memo from the numbers we have already computed. *"Python computes, model interprets"* — the model never invents numbers.

### Step 1 — verify the API key is available


In [ ]:
# Import os so we can read environment variables.
import os

# Read the Anthropic API key from the environment.
anth_key = os.environ.get("ANTHROPIC_API_KEY")

# If the key is missing, halt the notebook with a clear, actionable message.
if not anth_key:
    raise SystemExit(
        "ANTHROPIC_API_KEY is not set in your environment.\n"
        "Follow Step 4 of the repository README, close VS Code, reopen, and re-run this cell."
    )

# Confirm the key is set without printing its value.
print(f"ANTHROPIC_API_KEY is set. Key length: {len(anth_key)} characters.")


### Step 2 — create the Anthropic client


In [ ]:
# Import the official Anthropic Python SDK.
import anthropic

# Create the Anthropic client; it picks up ANTHROPIC_API_KEY from the environment automatically.
client = anthropic.Anthropic()

# Print a confirmation that the client object was created successfully.
print("Anthropic client ready.")


### Step 3 — define a Pydantic schema for the drivers plan

> **Callback to §3.12.** Pydantic gives us the JSON Schema for free. We write a regular Python class, and `.model_json_schema()` returns the JSON Schema dict Anthropic's tool-use feature expects. The same Pydantic class lets us re-wrap the validated response for type-safe access in the rest of the notebook.

The schema has three required string fields, matching the winemaker's brief:

- **`key_findings`** — what the drivers analysis actually shows.
- **`caveats`** — what assumptions, confounds, and data limitations the reader needs to know about before acting on the findings.
- **`data_that_would_strengthen_conclusion`** — what additional data we would collect or what additional analysis we would run to make the conclusion more robust.


In [ ]:
# Import Pydantic's BaseModel and Field helpers. Pydantic is installed in the `ailab` venv as an Anthropic SDK dependency.
from pydantic import BaseModel, Field

# Define the structure of a drivers plan as a Pydantic model.
class DriversPlan(BaseModel):
    key_findings: str = Field(
        description="The principal findings of the drivers analysis. Two to four sentences. "
                    "State the top drivers with their direction (positive or negative effect on quality) "
                    "and a rough characterisation of strength (e.g., 'strongest', 'second', 'minor'). "
                    "Use only the numbers given to you."
    )
    caveats: str = Field(
        description="The most important caveats. Two to four sentences. "
                    "Cover: correlation vs causation; the fact that the model explains only a fraction "
                    "of the spread; the integer-grid character of the panel score; and any other reasonable "
                    "honesty caveat. Do not invent specific numerical limitations."
    )
    data_that_would_strengthen_conclusion: str = Field(
        description="What additional data or analysis would make the conclusion more robust. "
                    "Two to four sentences. Concrete and operational — e.g., controlled fermentation "
                    "experiments, larger panel sizes, blind protocol changes, validation on a second vintage."
    )

# Print confirmation.
print("Pydantic schema DriversPlan defined.")


### Step 4 — call Anthropic with a forced tool

The mechanics are exactly the same as §3.12. We build a tool spec from our Pydantic schema and tell Anthropic the model **must** call it (`tool_choice` forces the call). The response comes back as a list of content blocks; we scan for the `tool_use` block and pull its `input` — that dictionary is the validated drivers plan.


In [ ]:
# Safety re-fit — make sure the three regressions this cell needs are available in scope.
# Everything below is already done earlier in the notebook (alcohol/density/quality arrays in §4.6/§4.9,
# sm_model in §4.9, multi_model & std_model in §4.14), but re-deriving them here is cheap and makes
# this cell robust to running cells out of order or restarting the kernel mid-way through.
alcohol = df["alcohol"].values
density = df["density"].values
quality = df["quality"].values
predictor_names = ["alcohol", "volatile acidity", "sulphates", "citric acid", "pH"]
sm_model    = sm.OLS(density, sm.add_constant(alcohol)).fit()
multi_model = sm.OLS(quality, sm.add_constant(df[predictor_names].values)).fit()
X_z_safe = (df[predictor_names] - df[predictor_names].mean()) / df[predictor_names].std(ddof=1)
y_z_safe = (quality - quality.mean()) / quality.std(ddof=1)
std_model = sm.OLS(y_z_safe, X_z_safe.values).fit()

# Build the Anthropic tool spec from our Pydantic schema.
# `model_json_schema()` returns a JSON Schema dict — exactly what Anthropic's `input_schema` field expects.
plan_tool = {
    "name": "submit_drivers_plan",
    "description": "Submit a complete drivers-analysis plan. You MUST call this tool with every required field filled in.",
    "input_schema": DriversPlan.model_json_schema(),
}

# Build the numbers summary the model is allowed to see (Python computes, model interprets).
top_ranked = (
    f"Standardised regression coefficients (z-scored predictors and outcome) on n = {len(quality)} red wines:\n"
    f"  alcohol           std beta = {std_model.params[0]:+.4f}  (strongest single driver, positive)\n"
    f"  volatile acidity  std beta = {std_model.params[1]:+.4f}  (second-strongest, negative)\n"
    f"  sulphates         std beta = {std_model.params[2]:+.4f}  (third, positive)\n"
    f"  pH                std beta = {std_model.params[4]:+.4f}  (minor)\n"
    f"  citric acid       std beta = {std_model.params[3]:+.4f}  (minor; sign flip from raw r — see §4.14)\n\n"
    f"Model-wide R^2 = {multi_model.rsquared:.4f}  (model accounts for ~{multi_model.rsquared*100:.0f}% of variance in panel quality scores)\n"
    f"Simple regression on alcohol alone: R^2 = {sm_model.rsquared:.4f}\n"
    f"All VIFs in the multiple regression are below 2.1 — no multicollinearity concern.\n"
    f"Outcome is panel score on integer grid 3..8 (most wines score 5 or 6).\n"
    f"Dataset is observational, not experimental: drivers analysis cannot make causal claims."
)

# Call Claude. `tool_choice` forces the model to call this specific tool — it cannot reply with plain text.
plan_response = client.messages.create(
    model="claude-haiku-4-5",                                          # same Haiku model used in §1.11, §2.13, §3.12
    max_tokens=1500,                                                   # generous cap; the tool input is a few short paragraphs
    system=(                                                           # Drivers Analyst persona
        "You are a Drivers Analyst Assistant for a mid-sized red-wine producer. "
        "You translate computed numbers from a drivers analysis into a structured, honest plan. "
        "You enforce three disciplines: (1) report only effects supported by the numbers given to you; "
        "(2) name correlation vs causation explicitly; (3) name any limitations the data carries. "
        "You never invent numbers; you focus on framing and honest interpretation."
    ),
    tools=[plan_tool],                                                 # the only tool the model is allowed to call
    tool_choice={"type": "tool", "name": "submit_drivers_plan"},       # force this specific tool — no free-text reply
    messages=[
        {
            "role": "user",
            "content": (
                "Here are the numbers from a drivers analysis of red wine panel quality "
                "(UCI Wine Quality, n = 1,599 wines, observational data):\n\n"
                + top_ranked
                + "\n\nPlease fill in the drivers plan. Use the numbers above and only those."
            )
        }
    ],
)

# Scan the response for the tool_use block. Because tool_choice forced our tool, exactly one such block exists.
plan_dict = None
for block in plan_response.content:
    if block.type == "tool_use":
        plan_dict = block.input    # this dict already conforms to our schema — Anthropic enforced it
        break

# Wrap the dict back into our Pydantic class for type-safe access.
plan = DriversPlan(**plan_dict)

# Show the plan field by field for easy reading.
for field_name, value in plan.model_dump().items():
    print(f"-- {field_name} --")
    print(f"  {value}\n")


**What we see.** Anthropic returns a tool call whose `input` is a JSON object with exactly the three fields our Pydantic schema specified, each filled with a short, structured statement. The `tool_choice` guarantee means no validation step is needed afterwards — the model literally cannot return something that fails the schema. Same generation-time guarantee as §3.12's analysis-plan call.

### Call 2 — Anthropic drafts the stakeholder memo paragraph

Now we hand the same computed numbers to Anthropic and ask for a short plain-text paragraph suitable for the *"Headline findings"* section of the one-page memo. Same *"Python computes, model interprets"* rule.


In [ ]:
# Call Anthropic to write the stakeholder paragraph for the memo.
memo_response = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=500,
    system=(
        "You translate analysis numbers into one short paragraph for a wine producer's stakeholder memo. "
        "Lead with the top three drivers and their direction. Name the model's R^2 so the reader knows how "
        "much of the panel-score variance is accounted for. Close with a one-sentence caveat about correlation vs causation. "
        "Use plain English. No mathematical symbols beyond plain percentages. "
        "Use ONLY the numbers given to you; never invent new numbers."
    ),
    messages=[
        {
            "role": "user",
            "content": (
                "Here are the numbers from the drivers analysis of red wine panel quality:\n\n"
                + top_ranked
                + "\n\nWrite ONE short paragraph (4 to 6 sentences) suitable for the 'Headline findings' "
                + "section of the head winemaker's stakeholder memo. Lead with the top three drivers, "
                + "name the R^2, and close with a correlation-not-causation caveat."
            )
        }
    ]
)

# Print Anthropic's paragraph.
print(memo_response.content[0].text)


**What we see.** A short, plain-English paragraph that leads with the top three drivers, names $R^2$, and closes with the causal caveat. The model **did not invent any numbers** — it was given the entire set it was allowed to use.

> **Mini-recap of §4.16.** Two API calls, one provider. Anthropic tool use with a Pydantic-derived schema **guarantees** the structured drivers plan at generation time. Plain-text Anthropic drafts the stakeholder paragraph from computed numbers. Same *"Python computes, model interprets"* discipline as Sessions 01–03; same single-provider architecture as §3.12. (Principle 20.)


***
## 4.17 The drivers plan and stakeholder memo — fully worked

It is Friday afternoon. Here are the two artifacts you would hand the head winemaker. Both are grounded entirely in numbers computed in this notebook.

---

### Artifact 1 — Drivers plan (pre-registered, three-field structure)

**Key findings.** The drivers analysis of $n = 1{,}599$ red wines identifies **alcohol** as the strongest positive driver of panel quality (standardised $\beta \approx +0.43$), **volatile acidity** as the strongest negative driver (standardised $\beta \approx -0.28$), and **sulphates** as a meaningful additional positive driver (standardised $\beta \approx +0.14$). `pH` and `citric acid` contribute only marginally in the multiple regression and may be flagged as low-priority for tuning. The five-predictor model accounts for about **34%** of the variance in panel quality scores; the simple regression on alcohol alone explains about **23%**.

**Caveats.** This is an **observational** analysis — the data comes from already-bottled wines, not from a controlled fermentation experiment. We are reporting **associations**, not **causation**. Three things in particular: (1) high-alcohol wines tend to come from riper grapes, so the alcohol effect may carry hidden contributions from other flavour compounds the panel rewards. (2) The outcome is an integer panel score on a 3–8 grid in this sample; this limits the resolution of any predictions. (3) About **two-thirds** of the variance in panel scores is **not** explained by these five predictors — much of what drives panel preference lives in compounds (or in panellist behaviour) we are not measuring.

**Data that would strengthen the conclusion.** First, a **controlled mini-experiment**: identify two fermentation conditions that, by chemistry, can be expected to produce wines of different alcohol levels but matched on the other measured variables; ferment a small batch of each; have the panel taste blind. This would let us upgrade the "alcohol drives quality" claim from an association to a tentative causal one. Second, **expand the panel**: three tasters per batch is the minimum; five-to-seven tasters would lower the noise floor in the outcome and probably lift $R^2$. Third, **validate on a second vintage** — the relationships found in this dataset should be reproduced before being acted on as a writing principle for next year's blends.

---

### Artifact 2 — Stakeholder memo

**To:** Head Winemaker
**From:** [Your name], Junior Analyst
**Re:** Drivers analysis — Wine Quality red, vintage analysis
**Date:** Friday, end of week 4

#### 1. What we analysed

Eleven physicochemical measurements taken during fermentation versus the panel quality score for 1,599 red wines from the UCI Wine Quality archive. The analysis is observational — these are already-bottled wines; we did not design any of the fermentation conditions.

#### 2. Headline findings

- **Alcohol** is the strongest single driver of panel quality: a one-standard-deviation increase in alcohol is associated with about a **+0.4 standard-deviation increase in quality**, all else equal. In the original units this is roughly **+0.33 quality points per +1% alcohol**.
- **Volatile acidity** is the strongest negative driver: higher volatile acidity is associated with lower panel scores. Standardised effect $\approx -0.28$.
- **Sulphates** contribute meaningfully and positively (standardised $\approx +0.14$).
- **pH** and **citric acid** contribute marginally; both are statistically significant but practically minor.
- The five-predictor model **accounts for about 34% of the variance** in panel scores; the simple alcohol-only model accounts for about 23%. Roughly two-thirds of panel-score variation remains unexplained.

#### 3. Limitations

- **Association, not causation.** The data is observational. We cannot say *"raising alcohol will raise quality"*; we can say *"in this dataset, higher-alcohol wines also scored higher"*. There are several plausible reasons for that, only one of which is *"alcohol itself"*.
- **Confounding.** Alcohol and density are strongly correlated ($r \approx -0.50$). The alcohol effect remains robust after controlling for density ($r_{\text{partial}} \approx +0.46$), which is reassuring — but the analysis cannot rule out other unmeasured confounds.
- **Outcome resolution.** The panel score is an integer on a 3–8 grid; this caps the precision of any prediction. Wines that *just barely* differ in chemistry will receive identical scores from the panel.
- **One vintage, one panel.** All wines in the archive were rated by the same panel under one protocol. Different panellists or different tasting conditions may yield different effect sizes.

#### 4. Recommendation

For the next vintage, **prioritise the alcohol and volatile-acidity levers**. Specifically: (a) avoid early harvests that produce wines below the regional alcohol mean; (b) tighten the cellar protocol that controls volatile acidity (cooler fermentation, faster racking, more careful sulphite management). The sulphates lever is worth tuning second. Treat the `pH` and `citric acid` findings as suggestive rather than actionable.

Before any of this is **acted on as a writing principle**, run a small controlled blind tasting of two-to-three intentionally-differentiated fermentation conditions, to convert the association into something closer to a causal claim. This is a half-vintage commitment, not a multi-year study.

#### 5. Open questions

- **Does the same driver ranking hold on the white-wine archive?** The same dataset family includes 4,898 white wines. A quick replication on `winequality-white.csv` would tell us whether these drivers generalise across wine styles.
- **Are there non-linear effects we are missing?** Pearson correlation and linear regression only see straight lines (§4.7). A wine with *"just enough"* sulphates may score higher than one with too little **or** too much — a relationship a linear model would systematically under-fit.
- **What is the contribution of compounds we are not measuring?** Two-thirds of the variance is still unaccounted for. A fuller chemistry panel (specific aroma compounds, tannin profiles) would let us widen the conversation.

---

That is your two-artifact deliverable. **Save your own memo as `_reports/session04_drivers_memo.md`** in your course directory. The instructor will review it next week.

> **Mini-recap of §4.17.** Two artifacts: a pre-registered drivers plan (generated with Anthropic tool use, grounded in the rigour of §§4.9–4.14) and a stakeholder memo (drafted with Anthropic on numbers we computed ourselves). Every claim traces back to a cell in this notebook. The structure prevents the most common drivers-memo mistakes — overstating raw correlations as causes, hiding caveats below the fold, and failing to specify what would strengthen the conclusion.


***
## 4.18 References — what to study to deepen this session

A focused list this week. **Pick three or four** of the StatQuest videos plus the Khan Academy *bivariate data* unit. The single most useful pair is the **Covariance + Pearson Correlation** duo from StatQuest, because once those two land properly, everything else in this notebook becomes simpler.

### StatQuest videos (YouTube)

| Video | What it clarifies |
|---|---|
| [Covariance, Clearly Explained](https://www.youtube.com/watch?v=qtaqvPAeEJY) | The intuition behind covariance as *"do these variables go up and down together?"*. Pairs with §4.6. |
| [Pearson's Correlation, Clearly Explained](https://www.youtube.com/watch?v=xZ_z8KWkhXE) | The unit-removal idea — exactly the *"standardise, then take covariance"* derivation in §4.7. Must-watch. |
| [R-squared, Clearly Explained](https://www.youtube.com/watch?v=2AQKmw14mHM) | The variance-decomposition framing for $R^2$ from §4.11. The cleanest visual explanation on the internet. |
| [The Main Ideas of Fitting a Line to Data (Least Squares)](https://www.youtube.com/watch?v=PaFPbb66DxQ) | Why we square the residuals and minimise the sum, in pictures. Pairs with §4.9's least-squares paragraph. |
| [Linear Regression, Clearly Explained](https://www.youtube.com/watch?v=nk2CQITm_eo) | Simple linear regression end-to-end, building on the least-squares video. Pairs with §4.9. |
| [Multiple Regression, Clearly Explained](https://www.youtube.com/watch?v=zITIFTsivN8) | Generalises the single-predictor model to several at once. The *"all else equal"* phrasing is foregrounded. Pairs with §4.14. |

### Khan Academy resources

- [Exploring bivariate numerical data unit](https://www.khanacademy.org/math/statistics-probability/describing-relationships-quantitative-data) — the structured course on scatter, correlation, and the regression line. The best place to consolidate everything from §§4.5–4.9.
- [Correlation coefficient review](https://www.khanacademy.org/math/statistics-probability/describing-relationships-quantitative-data/scatterplots-and-correlation/a/correlation-coefficient-review) — a tight summary of $r$, its sign, and its limits.
- [Calculating correlation coefficient r](https://www.khanacademy.org/math/statistics-probability/describing-relationships-quantitative-data/scatterplots-and-correlation/v/calculating-correlation-coefficient-r) — by-hand worked example. Pairs with §4.7.
- [Linear regression review](https://www.khanacademy.org/math/statistics-probability/describing-relationships-quantitative-data/more-on-regression/a/linear-regression-review) — slope, intercept, residual, predicted value — all in the same compact summary.
- [Regression line example](https://www.khanacademy.org/math/statistics-probability/describing-relationships-quantitative-data/more-on-regression/v/regression-line-example) — fitting a line by hand on a small dataset. Pairs with §4.9.
- [Covariance and the regression line](https://www.khanacademy.org/math/statistics-probability/describing-relationships-quantitative-data/more-on-regression/v/covariance-and-the-regression-line) — connects the covariance idea (§4.6) directly to the slope of the regression line.

> A good week: **the Covariance and Pearson Correlation StatQuest videos** for the §4.6–§4.7 backbone, **the R-squared StatQuest video** to lock in §4.11, **the Khan Academy bivariate-data unit** for end-to-end practice. That covers the spine of every later modelling session in this course.

See you in **Session 05**, where we leave continuous outcomes behind and move into **classification** — predicting a yes/no outcome from a set of predictors. The $R^2$ and standard-error machinery from this week will reappear in a slightly different guise (deviance and log-odds), but the conceptual primitive — *"predict, take residuals, see what is left"* — carries straight through.


<hr>

![](../_img/DK_Logo_White_150.png)

DataKolektiv, 2026.

[hello@datakolektiv.com](mailto:hello@datakolektiv.com)


<font size=1>License: [GPLv3](../LICENSE). This Notebook is free software: you can redistribute it and/or modify it under the terms of the GNU General Public License as published by the Free Software Foundation, either version 3 of the License, or (at your option) any later version. This Notebook is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY; without even the implied warranty of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE. See the GNU General Public License for more details.</font>
